# Tests de integración de `import_trips_from_dataframe()`

Notebook orientado a verificar el comportamiento end-to-end de OP-01 `import_trips`,
considerando resultado tabular, resultado operacional y trazabilidad.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "synthetic" 

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:
- imports generales,
- imports del módulo,
- helpers de testing reutilizables,
- smoke de imports,
- y configuración básica de display.

Aquí todavía no se prueban escenarios funcionales de `import_trips_from_dataframe()`;
solo se prepara el entorno de trabajo para las secciones siguientes.

### 0.1 Imports generales

In [2]:
import json
from pprint import pprint
import pandas as pd
import numpy as np

### 0.2 Imports del módulo 

In [3]:
from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec, TripSchemaEffective
from pylondrina.datasets import TripDataset
from pylondrina.reports import ImportReport, Issue
from pylondrina.errors import ImportError as PylondrinaImportError, SchemaError

### 0.3 Helpers de testing reutilizables

In [4]:
def show_df(df: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    """
    Muestra una vista rápida del DataFrame.
    """
    return df.head(n)

def issue_codes(report: ImportReport) -> list[str]:
    """
    Extrae los codes de los issues de un ImportReport.
    """
    return [issue.code for issue in report.issues]

def assert_issue_code(report: ImportReport, expected_code: str) -> None:
    """
    Verifica que un code exista dentro del report.
    """
    codes = issue_codes(report)
    assert expected_code in codes, (
        f"No se encontró el issue code esperado {expected_code!r}. "
        f"Codes presentes: {codes}"
    )

def assert_no_issue_code(report: ImportReport, forbidden_code: str) -> None:
    """
    Verifica que un code NO exista dentro del report.
    """
    codes = issue_codes(report)
    assert forbidden_code not in codes, (
        f"Se encontró el issue code prohibido {forbidden_code!r}. "
        f"Codes presentes: {codes}"
    )

def get_events(trips: TripDataset) -> list[dict]:
    """
    Retorna la lista de eventos del dataset, o lista vacía si no existe.
    """
    return list(trips.metadata.get("events", []))

def get_last_event(trips: TripDataset) -> dict:
    """
    Retorna el último evento registrado en metadata['events'].
    """
    events = get_events(trips)
    assert events, "El dataset no tiene eventos en metadata['events']."
    return events[-1]

def assert_last_event_op(trips: TripDataset, expected_op: str = "import_trips") -> None:
    """
    Verifica que el último evento corresponda a la operación esperada.
    """
    last_event = get_last_event(trips)
    assert last_event.get("op") == expected_op, (
        f"Se esperaba op={expected_op!r} y se obtuvo {last_event.get('op')!r}."
    )

def assert_tripdataset_and_report(result) -> tuple[TripDataset, ImportReport]:
    """
    Verifica que el retorno sea una tupla (TripDataset, ImportReport).
    """
    assert isinstance(result, tuple), f"Se esperaba tuple y se obtuvo {type(result)}."
    assert len(result) == 2, f"Se esperaban 2 elementos y se obtuvieron {len(result)}."
    trips, report = result
    assert isinstance(trips, TripDataset), f"Primer retorno inesperado: {type(trips)}."
    assert isinstance(report, ImportReport), f"Segundo retorno inesperado: {type(report)}."
    return trips, report

### 0.4 Configuración de display

In [5]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

**Cierre de la Sección 0**

El entorno del notebook quedó listo para:
- definir fixtures y objetos base,
- invocar generadores sintéticos externos,
- y comenzar luego las pruebas de integración sobre la función pública
`import_trips_from_dataframe(...)`.

## Sección 1. Fixtures y generadores sintéticos

Esta sección define la infraestructura reutilizable que se usará en las secciones
funcionales del notebook.

Aquí se fijan:

- schemas base para import,
- configuraciones típicas de `ImportOptions`,
- utilidades para invocar generadores sintéticos,
- un catálogo práctico DS-XX,
- y una guía breve de qué wrapper sirve para qué.

En esta etapa todavía no se prueban escenarios funcionales completos; solo se deja
preparado el entorno para las etapas siguientes.

### 1.1 Imports de generadores sintéticos

In [6]:
from scripts.synthetic_data.base_generator import generate_synthetic_trip_dataframe
from scripts.synthetic_data.gen_wrappers import (
    make_happy_path_minimal,
    make_happy_path_rich,
    make_h3_derivable,
    make_tier2_valid,
    make_tier2_mixed_invalid,
    make_multistage_valid,
    make_extended_domains,
    make_missing_required,
    make_duplicate_movement_id,
)

### 1.2 Schemas base reutilizables

#### 1.2.1 Schema base canónico de integración

Este debe reflejar:

- required canónicos del pool estricto para movements,
- algunos campos base categóricos útiles para tests posteriores,
- dominios explícitos para mode, purpose, day_type, time_period, user_gender.

In [7]:
BASE_TRIP_SCHEMA = TripSchema(
    version="1.1",
    fields={
        # required core
        "movement_id": FieldSpec("movement_id", "string", required=True),
        "user_id": FieldSpec("user_id", "string", required=True),
        "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
        "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
        "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
        "destination_latitude": FieldSpec("destination_latitude", "float", required=True),
        "origin_h3_index": FieldSpec("origin_h3_index", "string", required=True),
        "destination_h3_index": FieldSpec("destination_h3_index", "string", required=True),
        "origin_time_utc": FieldSpec("origin_time_utc", "datetime", required=False),
        "destination_time_utc": FieldSpec("destination_time_utc", "datetime", required=False),
        "trip_id": FieldSpec("trip_id", "string", required=True),
        "movement_seq": FieldSpec("movement_seq", "int", required=True),

        # base categorical
        "mode": FieldSpec(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=[
                    "walk", "bicycle", "scooter", "motorcycle", "car",
                    "taxi", "ride_hailing", "bus", "metro", "train", "other"
                ],
                extendable=True,
            ),
        ),
        "purpose": FieldSpec(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=[
                    "home", "work", "education", "shopping",
                    "errand", "health", "leisure", "transfer", "other"
                ],
                extendable=True,
            ),
        ),
        "day_type": FieldSpec(
            "day_type",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["weekday", "weekend", "holiday"],
                extendable=True,
            ),
        ),
        "time_period": FieldSpec(
            "time_period",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["night", "morning", "midday", "afternoon", "evening"],
                extendable=True,
            ),
        ),
        "user_gender": FieldSpec(
            "user_gender",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["female", "male", "other", "unknown"],
                extendable=True,
            ),
        ),

        # base non-categorical
        "origin_time_local_hhmm": FieldSpec("origin_time_local_hhmm", "string", required=False),
        "destination_time_local_hhmm": FieldSpec("destination_time_local_hhmm", "string", required=False),
        "origin_municipality": FieldSpec("origin_municipality", "string", required=False),
        "destination_municipality": FieldSpec("destination_municipality", "string", required=False),
        "trip_weight": FieldSpec("trip_weight", "float", required=False),
    },
    required=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
    ],
)

#### 1.2.2 Schema para H3 derivable

In [8]:
H3_DERIVABLE_SCHEMA = BASE_TRIP_SCHEMA

### 1.2.3 Schema con dominio no extendable

In [9]:
NON_EXTENDABLE_DOMAIN_SCHEMA = TripSchema(
    version="1.1",
    fields={
        **BASE_TRIP_SCHEMA.fields,
        "mode": FieldSpec(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=[
                    "walk", "bicycle", "scooter", "motorcycle", "car",
                    "taxi", "ride_hailing", "bus", "metro", "train", "other", "unknown"
                ],
                extendable=False,
            ),
        ),
    },
    required=list(BASE_TRIP_SCHEMA.required),
)

### 1.4 ImportOptions base reutilizables

Esto debe reflejar las decisiones cerradas de v1.1 sobre ImportOptions: selected_fields=None conserva lo presente/derivable; [] conserva solo required; single_stage=True permite derivar trip_id y fijar movement_seq=0; source_timezone aplica solo a Tier 1 naive.

In [10]:
IMPORT_OPTIONS_DEFAULT = ImportOptions()

IMPORT_OPTIONS_STRICT = ImportOptions(
    strict=True,
)

IMPORT_OPTIONS_SINGLE_STAGE = ImportOptions(
    single_stage=True,
)

IMPORT_OPTIONS_KEEP_ONLY_REQUIRED = ImportOptions(
    selected_fields=[],
    keep_extra_fields=False,
)

IMPORT_OPTIONS_KEEP_ALL_WITH_EXTRAS = ImportOptions(
    selected_fields=None,
    keep_extra_fields=True,
)

IMPORT_OPTIONS_STRICT_DOMAINS = ImportOptions(
    strict_domains=True,
)

IMPORT_OPTIONS_TIER1_SANTIAGO = ImportOptions(
    source_timezone="America/Santiago",
)

### 1.5 Provenance base reutilizable

In [11]:
BASE_PROVENANCE = {
    "source": {
        "name": "synthetic",
        "entity": "trips",
        "version": "integration-tests-v1",
    },
    "notes": ["dataset sintético para tests de integración OP-01"],
}

### 1.6 Utilidades

In [12]:
def describe_df(df: pd.DataFrame) -> dict:
    """
    Resumen corto para inspección rápida de datasets sintéticos.
    """
    return {
        "shape": df.shape,
        "columns": list(df.columns),
        "dtypes": {c: str(t) for c, t in df.dtypes.items()},
    }

### 1.7 Catálogo práctico DS-XX

| DS | Propósito principal | Etapas/Secciones | Nivel inicial sugerido | Wrapper / estrategia |
|---|---|---|---|---|
| DS-01 | Happy path canónico | Sección 2, Sección 6 | Controlado | `make_happy_path_minimal` |
| DS-02 | Happy path con correspondencias | Sección 2, Sección 6 | Controlado | `make_happy_path_minimal(..., field_correspondence=...)` o base |
| DS-03 | Single-stage | Sección 2 | Controlado | wrapper/base con omisión de `trip_id` y `movement_seq` |
| DS-04 | Tier 2 | Sección 2, 4, 6 | Controlado | `make_tier2_valid`, `make_tier2_mixed_invalid` |
| DS-05 | Espacial con H3 derivable | Sección 2, 4 | Controlado | `make_h3_derivable` |
| DS-06 | Dominio extendable | Sección 4, 6 | Controlado/Rico | `make_extended_domains` |
| DS-07 | Dominio no extendable | Sección 4 | Controlado | `make_extended_domains` + schema no extendable |
| DS-08 | Tier 1 parcial | Sección 4 | Controlado | función base |
| DS-09 | Coordenadas mixtas | Sección 4 | Controlado | función base |
| DS-10 | Missing required fatal | Sección 3 | Controlado | `make_missing_required` |
| DS-11 | Mapping inválido | Sección 3 | Controlado | función base o test manual con mapping malo |
| DS-12 | `movement_id` duplicado | Sección 3 | Controlado | `make_duplicate_movement_id` |
| DS-13 | H3 no materializable | Sección 3 | Controlado | función base |
| DS-14 | Selección final de columnas | Sección 5 | Controlado/Rico | `make_happy_path_rich` |
| DS-15 | Trazabilidad rica | Sección 6 | Rico | `make_happy_path_rich` |

### 1.8 Guía breve de wrappers

- `make_happy_path_minimal`:
  caso feliz más simple posible; required canónicos, Tier 1 limpio, H3 presentes.

- `make_happy_path_rich`:
  caso feliz más parecido a una fuente real moderadamente completa; útil para trazabilidad y salida observable.

- `make_h3_derivable`:
  omite explícitamente `origin_h3_index` y `destination_h3_index`; pensado para probar derivación H3.

- `make_tier2_valid`:
  temporalidad Tier 2 válida (`HH:MM`).

- `make_tier2_mixed_invalid`:
  mezcla de HH:MM válidos e inválidos; útil para calidad no fatal.

- `make_multistage_valid`:
  varios movements pueden compartir `trip_id`; útil cuando se quiera revisar estructura multi-etapa.

- `make_extended_domains`:
  mezcla dominios canónicos con valores extendidos controlados.

- `make_missing_required`:
  dataset deliberadamente inválido por omisión de required.

- `make_duplicate_movement_id`:
  dataset deliberadamente inválido por duplicación de `movement_id`.

**Cierre de la Sección 1**

Quedó preparada la infraestructura del notebook para las etapas funcionales siguientes:

- schemas reutilizables,
- `ImportOptions` base,
- acceso uniforme a generadores sintéticos,
- y catálogo práctico DS-XX.

A partir de la siguiente etapa ya corresponde definir tests concretos por bloque,
junto con la llamada exacta al wrapper o generador que produzca el dataset requerido.

## Sección 2. Caminos felices mínimos

En esta sección se verifica que `import_trips_from_dataframe(...)` complete correctamente
casos end-to-end donde el dataset es construible y no debe abortar.

Cada caso revisa simultáneamente:

- resultado tabular,
- resultado operacional,
- trazabilidad.

### A1. Happy path canónico

In [13]:
df_a1 = make_happy_path_minimal(
    filas=6,
    seed=42,
)
display(df_a1)

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index
0,m00000,u0000,t00000,0,-70.745304,-33.374755,-70.772542,-33.400534,2026-03-04T17:37:00Z,2026-03-04T19:28:00Z,88b2c5576dfffff,88b2c5508bfffff
1,m00001,u0002,t00001,0,-70.572060,-33.606083,-70.523159,-33.595020,2026-03-01T13:21:00Z,2026-03-01T14:51:00Z,88b2c57315fffff,88b2c57207fffff
2,m00002,u0001,t00002,0,-70.582221,-33.554174,-70.588402,-33.582941,2026-03-05T01:49:00Z,2026-03-05T02:36:00Z,88b2c509a1fffff,88b2c57347fffff
3,m00003,u0001,t00003,0,-70.653397,-33.439773,-70.670530,-33.413419,2026-03-02T07:55:00Z,2026-03-02T09:51:00Z,88b2c55411fffff,88b2c55443fffff
4,m00004,u0001,t00004,0,-70.547276,-33.475299,-70.561361,-33.476797,2026-03-06T10:53:00Z,2026-03-06T11:45:00Z,88b2c50b13fffff,88b2c50b3bfffff
5,m00005,u0002,t00005,0,-70.613249,-33.451344,-70.591957,-33.456890,2026-03-06T00:45:00Z,2026-03-06T01:27:00Z,88b2c554d9fffff,88b2c50b67fffff


In [14]:
trips_a1, report_a1 = import_trips_from_dataframe(
    df_a1,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_a1",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)
display(trips_a1.data)
display(report_a1.issues)
display(trips_a1.metadata)

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index
0,m00000,u0000,t00000,0,-70.745304,-33.374755,-70.772542,-33.400534,2026-03-04 17:37:00+00:00,2026-03-04 19:28:00+00:00,88b2c5576dfffff,88b2c5508bfffff
1,m00001,u0002,t00001,0,-70.572060,-33.606083,-70.523159,-33.595020,2026-03-01 13:21:00+00:00,2026-03-01 14:51:00+00:00,88b2c57315fffff,88b2c57207fffff
2,m00002,u0001,t00002,0,-70.582221,-33.554174,-70.588402,-33.582941,2026-03-05 01:49:00+00:00,2026-03-05 02:36:00+00:00,88b2c509a1fffff,88b2c57347fffff
3,m00003,u0001,t00003,0,-70.653397,-33.439773,-70.670530,-33.413419,2026-03-02 07:55:00+00:00,2026-03-02 09:51:00+00:00,88b2c55411fffff,88b2c55443fffff
4,m00004,u0001,t00004,0,-70.547276,-33.475299,-70.561361,-33.476797,2026-03-06 10:53:00+00:00,2026-03-06 11:45:00+00:00,88b2c50b13fffff,88b2c50b3bfffff
5,m00005,u0002,t00005,0,-70.613249,-33.451344,-70.591957,-33.456890,2026-03-06 00:45:00+00:00,2026-03-06 01:27:00+00:00,88b2c554d9fffff,88b2c50b67fffff


[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'mode' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='mode', source_field=None, row_count=None, details={'field': 'mode', 'source_columns': ['movement_id', 'user_id', 'trip_id', 'movement_seq', 'origin_longitude', 'origin_latitude', 'destination_longitude', 'destination_latitude', 'origin_time_utc', 'destination_time_utc', 'origin_h3_index', 'destination_h3_index'], 'field_correspondence_used': False, 'action': 'omit_optional'}),
 Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'purpose' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='purpose', source_field=None, row_count=None, details={'field': 'purpose', 'source_columns': ['movement_id', 'user_id', 'trip_id', 'movement_seq', 'origin_longitude', 'origin_latitude', 'destination_longitude', 'destination_latitude', 'origin_time_utc'

{'dataset_id': 'tripds_c4c3559990bd46c7b1d69ef97c798ff0',
 'is_validated': False,
 'schema': {'version': '1.1',
  'fields': {'movement_id': {'name': 'movement_id',
    'dtype': 'string',
    'required': True,
    'constraints': None,
    'domain': None},
   'user_id': {'name': 'user_id',
    'dtype': 'string',
    'required': True,
    'constraints': None,
    'domain': None},
   'origin_longitude': {'name': 'origin_longitude',
    'dtype': 'float',
    'required': True,
    'constraints': None,
    'domain': None},
   'origin_latitude': {'name': 'origin_latitude',
    'dtype': 'float',
    'required': True,
    'constraints': None,
    'domain': None},
   'destination_longitude': {'name': 'destination_longitude',
    'dtype': 'float',
    'required': True,
    'constraints': None,
    'domain': None},
   'destination_latitude': {'name': 'destination_latitude',
    'dtype': 'float',
    'required': True,
    'constraints': None,
    'domain': None},
   'origin_h3_index': {'name': 'orig

In [15]:
# retorno básico
assert isinstance(trips_a1, TripDataset)
assert isinstance(report_a1, ImportReport)
assert report_a1.ok is True

# tabular
assert len(trips_a1.data) == 6
for col in [
    "movement_id", "user_id",
    "origin_longitude", "origin_latitude",
    "destination_longitude", "destination_latitude",
    "origin_h3_index", "destination_h3_index",
    "trip_id", "movement_seq",
]:
    assert col in trips_a1.data.columns

# operacional
assert report_a1.field_correspondence == {}
assert report_a1.value_correspondence == {}

# trazabilidad
assert "dataset_id" in trips_a1.metadata
assert trips_a1.metadata["is_validated"] is False
assert "events" in trips_a1.metadata
assert trips_a1.metadata["events"][-1]["op"] == "import_trips"
assert "schema_effective" in trips_a1.metadata
assert "temporal" in trips_a1.metadata
assert trips_a1.metadata["temporal"]["tier"] == "tier_1"

### A2. Happy path con correspondencias

Aquí hay dos cosas distintas:

- correspondencia de campos,
- correspondencia de valores.

Para que el caso sea realmente útil, conviene fijar ambas de forma determinista.

In [16]:
field_corr_a2 = {
    "movement_id": "id_mov_fuente",
    "user_id": "id_persona_fuente",
    "origin_longitude": "lon_o_fuente",
    "origin_latitude": "lat_o_fuente",
    "destination_longitude": "lon_d_fuente",
    "destination_latitude": "lat_d_fuente",
    "origin_h3_index": "h3_o_fuente",
    "destination_h3_index": "h3_d_fuente",
    "origin_time_utc": "t_origen_fuente",
    "destination_time_utc": "t_destino_fuente",
    "trip_id": "id_viaje_fuente",
    "movement_seq": "seq_fuente",
    "mode": "modo_fuente",
    "purpose": "proposito_fuente",
}

df_a2 = make_happy_path_minimal(
    filas=6,
    seed=42,
    base_fields=["mode", "purpose"],
    field_correspondence=field_corr_a2,
)

# Edicion explicita para no epender de valores aleatorios del generador
df_a2["modo_fuente"] = ["A PIE", "BUS", "AUTO", "METRO", "A PIE", "BUS"]
df_a2["proposito_fuente"] = ["TRABAJO", "ESTUDIO", "HOGAR", "COMPRAS", "TRABAJO", "HOGAR"]

value_corr_a2 = {
    "mode": {
        "A PIE": "walk",
        "BUS": "bus",
        "AUTO": "car",
        "METRO": "metro",
    },
    "purpose": {
        "TRABAJO": "work",
        "ESTUDIO": "education",
        "HOGAR": "home",
        "COMPRAS": "shopping",
    },
}

In [17]:
trips_a2, report_a2 = import_trips_from_dataframe(
    df_a2,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_a2",
    options=ImportOptions(),
    field_correspondence=field_corr_a2,
    value_correspondence=value_corr_a2,
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [18]:
assert report_a2.ok is True

# tabular
assert "mode" in trips_a2.data.columns
assert "purpose" in trips_a2.data.columns
assert "modo_fuente" not in trips_a2.data.columns
assert "proposito_fuente" not in trips_a2.data.columns
assert set(trips_a2.data["mode"].dropna().unique()) <= {"walk", "bus", "car", "metro"}
assert set(trips_a2.data["purpose"].dropna().unique()) <= {"work", "education", "home", "shopping"}

# operacional
assert report_a2.field_correspondence == field_corr_a2
assert report_a2.value_correspondence == value_corr_a2

# trazabilidad
assert trips_a2.field_correspondence == field_corr_a2
assert trips_a2.value_correspondence == value_corr_a2
assert "mappings" in trips_a2.metadata
assert "field_correspondence" in trips_a2.metadata["mappings"]
assert "value_correspondence" in trips_a2.metadata["mappings"]

In [19]:
display(df_a2)
display(trips_a2.data)

,id_mov_fuente,id_persona_fuente,id_viaje_fuente,seq_fuente,lon_o_fuente,lat_o_fuente,lon_d_fuente,lat_d_fuente,t_origen_fuente,t_destino_fuente,h3_o_fuente,h3_d_fuente,modo_fuente,proposito_fuente
0,m00000,u0000,t00000,0,-70.745304,-33.374755,-70.772542,-33.400534,2026-03-04T17:37:00Z,2026-03-04T19:28:00Z,88b2c5576dfffff,88b2c5508bfffff,A PIE,TRABAJO
1,m00001,u0002,t00001,0,-70.572060,-33.606083,-70.523159,-33.595020,2026-03-01T13:21:00Z,2026-03-01T14:51:00Z,88b2c57315fffff,88b2c57207fffff,BUS,ESTUDIO
2,m00002,u0001,t00002,0,-70.582221,-33.554174,-70.588402,-33.582941,2026-03-05T01:49:00Z,2026-03-05T02:36:00Z,88b2c509a1fffff,88b2c57347fffff,AUTO,HOGAR
3,m00003,u0001,t00003,0,-70.653397,-33.439773,-70.670530,-33.413419,2026-03-02T07:55:00Z,2026-03-02T09:51:00Z,88b2c55411fffff,88b2c55443fffff,METRO,COMPRAS
4,m00004,u0001,t00004,0,-70.547276,-33.475299,-70.561361,-33.476797,2026-03-06T10:53:00Z,2026-03-06T11:45:00Z,88b2c50b13fffff,88b2c50b3bfffff,A PIE,TRABAJO
5,m00005,u0002,t00005,0,-70.613249,-33.451344,-70.591957,-33.456890,2026-03-06T00:45:00Z,2026-03-06T01:27:00Z,88b2c554d9fffff,88b2c50b67fffff,BUS,HOGAR


,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,mode,purpose
0,m00000,u0000,t00000,0,-70.745304,-33.374755,-70.772542,-33.400534,2026-03-04 17:37:00+00:00,2026-03-04 19:28:00+00:00,88b2c5576dfffff,88b2c5508bfffff,walk,work
1,m00001,u0002,t00001,0,-70.572060,-33.606083,-70.523159,-33.595020,2026-03-01 13:21:00+00:00,2026-03-01 14:51:00+00:00,88b2c57315fffff,88b2c57207fffff,bus,education
2,m00002,u0001,t00002,0,-70.582221,-33.554174,-70.588402,-33.582941,2026-03-05 01:49:00+00:00,2026-03-05 02:36:00+00:00,88b2c509a1fffff,88b2c57347fffff,car,home
3,m00003,u0001,t00003,0,-70.653397,-33.439773,-70.670530,-33.413419,2026-03-02 07:55:00+00:00,2026-03-02 09:51:00+00:00,88b2c55411fffff,88b2c55443fffff,metro,shopping
4,m00004,u0001,t00004,0,-70.547276,-33.475299,-70.561361,-33.476797,2026-03-06 10:53:00+00:00,2026-03-06 11:45:00+00:00,88b2c50b13fffff,88b2c50b3bfffff,walk,work
5,m00005,u0002,t00005,0,-70.613249,-33.451344,-70.591957,-33.456890,2026-03-06 00:45:00+00:00,2026-03-06 01:27:00+00:00,88b2c554d9fffff,88b2c50b67fffff,bus,home


### A3. Happy path con single_stage=True

In [20]:
df_a3 = generate_synthetic_trip_dataframe(
    filas=6,
    seed=42,
    tier_temporal="tier_1",
    tier1_datetime_format="utc_string_z",
    coord_format="numeric",
    h3_mode="provided_valid",
    trip_structure="single_stage_like",
    base_fields=["mode", "purpose"],
    omit_required_fields=["trip_id", "movement_seq"],
)

In [21]:
trips_a3, report_a3 = import_trips_from_dataframe(
    df_a3,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_a3",
    options=ImportOptions(single_stage=True),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [22]:
assert report_a3.ok is True

# tabular
assert "trip_id" in trips_a3.data.columns
assert "movement_seq" in trips_a3.data.columns
assert (trips_a3.data["movement_seq"] == 0).all()
assert trips_a3.data["trip_id"].notna().all()
assert trips_a3.data["movement_id"].notna().all()

# operacional
assert len(trips_a3.data) == 6

# trazabilidad
assert "schema_effective" in trips_a3.metadata
assert trips_a3.metadata["events"][-1]["parameters"]["single_stage"] is True

In [23]:
display(df_a3)
display(trips_a3.data)

,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,mode,purpose
0,m00000,u0000,-70.745304,-33.374755,-70.772542,-33.400534,2026-03-04T17:37:00Z,2026-03-04T19:28:00Z,88b2c5576dfffff,88b2c5508bfffff,taxi,health
1,m00001,u0002,-70.572060,-33.606083,-70.523159,-33.595020,2026-03-01T13:21:00Z,2026-03-01T14:51:00Z,88b2c57315fffff,88b2c57207fffff,bicycle,leisure
2,m00002,u0001,-70.582221,-33.554174,-70.588402,-33.582941,2026-03-05T01:49:00Z,2026-03-05T02:36:00Z,88b2c509a1fffff,88b2c57347fffff,bus,home
3,m00003,u0001,-70.653397,-33.439773,-70.670530,-33.413419,2026-03-02T07:55:00Z,2026-03-02T09:51:00Z,88b2c55411fffff,88b2c55443fffff,taxi,education
4,m00004,u0001,-70.547276,-33.475299,-70.561361,-33.476797,2026-03-06T10:53:00Z,2026-03-06T11:45:00Z,88b2c50b13fffff,88b2c50b3bfffff,motorcycle,leisure
5,m00005,u0002,-70.613249,-33.451344,-70.591957,-33.456890,2026-03-06T00:45:00Z,2026-03-06T01:27:00Z,88b2c554d9fffff,88b2c50b67fffff,scooter,transfer


,movement_id,trip_id,movement_seq,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,mode,purpose
0,m00000,m00000,0,u0000,-70.745304,-33.374755,-70.772542,-33.400534,2026-03-04 17:37:00+00:00,2026-03-04 19:28:00+00:00,88b2c5576dfffff,88b2c5508bfffff,taxi,health
1,m00001,m00001,0,u0002,-70.572060,-33.606083,-70.523159,-33.595020,2026-03-01 13:21:00+00:00,2026-03-01 14:51:00+00:00,88b2c57315fffff,88b2c57207fffff,bicycle,leisure
2,m00002,m00002,0,u0001,-70.582221,-33.554174,-70.588402,-33.582941,2026-03-05 01:49:00+00:00,2026-03-05 02:36:00+00:00,88b2c509a1fffff,88b2c57347fffff,bus,home
3,m00003,m00003,0,u0001,-70.653397,-33.439773,-70.670530,-33.413419,2026-03-02 07:55:00+00:00,2026-03-02 09:51:00+00:00,88b2c55411fffff,88b2c55443fffff,taxi,education
4,m00004,m00004,0,u0001,-70.547276,-33.475299,-70.561361,-33.476797,2026-03-06 10:53:00+00:00,2026-03-06 11:45:00+00:00,88b2c50b13fffff,88b2c50b3bfffff,motorcycle,leisure
5,m00005,m00005,0,u0002,-70.613249,-33.451344,-70.591957,-33.456890,2026-03-06 00:45:00+00:00,2026-03-06 01:27:00+00:00,88b2c554d9fffff,88b2c50b67fffff,scooter,transfer


### A4. Happy path Tier 2

In [24]:
df_a4 = make_tier2_valid(
    filas=6,
    seed=42,
)

In [25]:
trips_a4, report_a4 = import_trips_from_dataframe(
    df_a4,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_a4",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [26]:
assert report_a4.ok is True

# tabular
assert "origin_time_local_hhmm" in trips_a4.data.columns
assert "destination_time_local_hhmm" in trips_a4.data.columns
assert trips_a4.data["origin_time_local_hhmm"].notna().all()
assert trips_a4.data["destination_time_local_hhmm"].notna().all()

# operacional
assert len(trips_a4.data) == 6

# trazabilidad
assert "temporal" in trips_a4.metadata
assert trips_a4.metadata["temporal"]["tier"] == "tier_2"
assert "origin_time_local_hhmm" in trips_a4.metadata["temporal"]["fields_present"]
assert "destination_time_local_hhmm" in trips_a4.metadata["temporal"]["fields_present"]

In [27]:
display(df_a4)
display(trips_a4.data)

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_local_hhmm,destination_time_local_hhmm,origin_h3_index,destination_h3_index,mode,purpose
0,m00000,u0000,t00000,0,-70.745304,-33.374755,-70.772542,-33.400534,17:37,19:28,88b2c5576dfffff,88b2c5508bfffff,taxi,health
1,m00001,u0002,t00001,0,-70.572060,-33.606083,-70.523159,-33.595020,13:21,14:51,88b2c57315fffff,88b2c57207fffff,bicycle,leisure
2,m00002,u0001,t00002,0,-70.582221,-33.554174,-70.588402,-33.582941,01:49,02:36,88b2c509a1fffff,88b2c57347fffff,bus,home
3,m00003,u0001,t00003,0,-70.653397,-33.439773,-70.670530,-33.413419,07:55,09:51,88b2c55411fffff,88b2c55443fffff,taxi,education
4,m00004,u0001,t00004,0,-70.547276,-33.475299,-70.561361,-33.476797,10:53,11:45,88b2c50b13fffff,88b2c50b3bfffff,motorcycle,leisure
5,m00005,u0002,t00005,0,-70.613249,-33.451344,-70.591957,-33.456890,00:45,01:27,88b2c554d9fffff,88b2c50b67fffff,scooter,transfer


,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_local_hhmm,destination_time_local_hhmm,origin_h3_index,destination_h3_index,mode,purpose
0,m00000,u0000,t00000,0,-70.745304,-33.374755,-70.772542,-33.400534,17:37,19:28,88b2c5576dfffff,88b2c5508bfffff,taxi,health
1,m00001,u0002,t00001,0,-70.572060,-33.606083,-70.523159,-33.595020,13:21,14:51,88b2c57315fffff,88b2c57207fffff,bicycle,leisure
2,m00002,u0001,t00002,0,-70.582221,-33.554174,-70.588402,-33.582941,01:49,02:36,88b2c509a1fffff,88b2c57347fffff,bus,home
3,m00003,u0001,t00003,0,-70.653397,-33.439773,-70.670530,-33.413419,07:55,09:51,88b2c55411fffff,88b2c55443fffff,taxi,education
4,m00004,u0001,t00004,0,-70.547276,-33.475299,-70.561361,-33.476797,10:53,11:45,88b2c50b13fffff,88b2c50b3bfffff,motorcycle,leisure
5,m00005,u0002,t00005,0,-70.613249,-33.451344,-70.591957,-33.456890,00:45,01:27,88b2c554d9fffff,88b2c50b67fffff,scooter,transfer


### A5. Happy path con derivación H3

In [28]:
df_a5 = make_h3_derivable(
    filas=8,
    seed=42,
)

In [29]:
trips_a5, report_a5 = import_trips_from_dataframe(
    df_a5,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_a5",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [30]:
assert report_a5.ok is True

# tabular
assert "origin_h3_index" in trips_a5.data.columns
assert "destination_h3_index" in trips_a5.data.columns
assert trips_a5.data["origin_h3_index"].notna().all()
assert trips_a5.data["destination_h3_index"].notna().all()

# operacional
assert len(trips_a5.data) == 8

# trazabilidad
assert "h3" in trips_a5.metadata
assert trips_a5.metadata["h3"]["resolution"] == 8
assert trips_a5.metadata["events"][-1]["parameters"]["h3_resolution"] == 8

In [31]:
display(df_a5)
display(trips_a5.data)

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,mode,purpose
0,m00000,u0000,t00000,0,-70.653397,-33.606083,-70.636888,-33.611629,2026-03-04T11:46:00Z,2026-03-04T13:39:00Z,scooter,health
1,m00001,u0003,t00001,0,-70.547276,-33.554174,-70.530043,-33.574602,2026-03-02T03:49:00Z,2026-03-02T04:44:00Z,bus,errand
2,m00002,u0002,t00002,0,-70.613249,-33.439773,-70.527583,-33.403097,2026-03-06T01:19:00Z,2026-03-06T01:42:00Z,bus,errand
3,m00003,u0001,t00003,0,-70.745929,-33.475299,-70.762186,-33.479935,2026-03-04T13:55:00Z,2026-03-04T15:35:00Z,bicycle,health
4,m00004,u0001,t00004,0,-70.623125,-33.451344,-70.643615,-33.464194,2026-03-03T13:28:00Z,2026-03-03T14:45:00Z,train,home
5,m00005,u0003,t00005,0,-70.755888,-33.518244,-70.788439,-33.528808,2026-03-02T20:07:00Z,2026-03-02T21:32:00Z,scooter,work
6,m00006,u0000,t00006,0,-70.572155,-33.379648,-70.547516,-33.363679,2026-03-05T04:50:00Z,2026-03-05T05:06:00Z,metro,education
7,m00007,u0002,t00007,0,-70.664993,-33.387777,-70.619834,-33.376813,2026-03-05T22:31:00Z,2026-03-05T23:11:00Z,walk,work


,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,mode,purpose,origin_h3_index,destination_h3_index
0,m00000,u0000,t00000,0,-70.653397,-33.606083,-70.636888,-33.611629,2026-03-04 11:46:00+00:00,2026-03-04 13:39:00+00:00,scooter,health,88b2c54481fffff,88b2c5449dfffff
1,m00001,u0003,t00001,0,-70.547276,-33.554174,-70.530043,-33.574602,2026-03-02 03:49:00+00:00,2026-03-02 04:44:00+00:00,bus,errand,88b2c50995fffff,88b2c5726bfffff
2,m00002,u0002,t00002,0,-70.613249,-33.439773,-70.527583,-33.403097,2026-03-06 01:19:00+00:00,2026-03-06 01:42:00+00:00,bus,errand,88b2c556b1fffff,88b2c519d7fffff
3,m00003,u0001,t00003,0,-70.745929,-33.475299,-70.762186,-33.479935,2026-03-04 13:55:00+00:00,2026-03-04 15:35:00+00:00,bicycle,health,88b2c542d9fffff,88b2c542c3fffff
4,m00004,u0001,t00004,0,-70.623125,-33.451344,-70.643615,-33.464194,2026-03-03 13:28:00+00:00,2026-03-03 14:45:00+00:00,train,home,88b2c554ddfffff,88b2c5548dfffff
5,m00005,u0003,t00005,0,-70.755888,-33.518244,-70.788439,-33.528808,2026-03-02 20:07:00+00:00,2026-03-02 21:32:00+00:00,scooter,work,88b2c5476dfffff,88b2c540c3fffff
6,m00006,u0000,t00006,0,-70.572155,-33.379648,-70.547516,-33.363679,2026-03-05 04:50:00+00:00,2026-03-05 05:06:00+00:00,metro,education,88b2c51915fffff,88b2c5182dfffff
7,m00007,u0002,t00007,0,-70.664993,-33.387777,-70.619834,-33.376813,2026-03-05 22:31:00+00:00,2026-03-05 23:11:00+00:00,walk,work,88b2c5562bfffff,88b2c55653fffff


### A1R. Happy path canónico rico

In [32]:
df_a1r = make_happy_path_rich(
    filas=15,
    seed=42,
)

trips_a1r, report_a1r = import_trips_from_dataframe(
    df_a1r,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_a1r",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

assert report_a1r.ok is True
assert len(trips_a1r.data) == 15
assert "dataset_id" in trips_a1r.metadata
assert "events" in trips_a1r.metadata
assert "schema_effective" in trips_a1r.metadata

### A4R. Tier 2 rico

In [33]:
df_a4r = make_tier2_valid(
    filas=50,
    seed=42,
    extra_columns=["household_id", "travel_time_min"],
)
trips_a4r, report_a4r = import_trips_from_dataframe(
    df_a4r,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_a4r",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

display(trips_a4r.data.head(10))

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_local_hhmm,destination_time_local_hhmm,origin_h3_index,destination_h3_index,mode,purpose,household_id,travel_time_min
0,m00000,u0002,t00000,0,-70.590951,-33.478171,-70.584048,-33.467465,11:48,13:29,88b2c50949fffff,88b2c50b2bfffff,metro,leisure,H00023,29.1
1,m00001,u0019,t00001,0,-70.702725,-33.407415,-70.639562,-33.363516,04:24,06:02,88b2c55717fffff,88b2c55643fffff,car,leisure,H00023,76.1
2,m00002,u0016,t00002,0,-70.644146,-33.420764,-70.637746,-33.456427,07:05,09:00,88b2c5545bfffff,88b2c554c7fffff,bus,other,H00009,14.0
3,m00003,u0010,t00003,0,-70.597441,-33.416981,-70.602186,-33.436174,00:19,00:36,88b2c5568bfffff,88b2c556bbfffff,bus,transfer,H00019,89.9
4,m00004,u0010,t00004,0,-70.690935,-33.415534,-70.679502,-33.443332,09:52,11:38,88b2c55469fffff,88b2c5543dfffff,scooter,errand,H00015,61.6
5,m00005,u0021,t00005,0,-70.614322,-33.278668,-70.562082,-33.290362,10:08,10:20,88b2c5cea9fffff,88b2c51b4dfffff,scooter,errand,H00011,60.3
6,m00006,u0002,t00006,0,-70.726193,-33.482513,-70.717417,-33.523814,12:29,13:28,88b2c555a9fffff,88b2c54755fffff,walk,education,H00003,12.7
7,m00007,u0017,t00007,0,-70.696305,-33.490979,-70.712742,-33.471925,18:44,19:57,88b2c555b9fffff,88b2c555ebfffff,bicycle,health,H00009,81.2
8,m00008,u0005,t00008,0,-70.698174,-33.515102,-70.653922,-33.521768,10:02,10:48,88b2c5475bfffff,88b2c54603fffff,car,work,H00016,7.5
9,m00009,u0002,t00009,0,-70.779584,-33.400722,-70.762434,-33.444846,21:07,21:28,88b2c55081fffff,88b2c55529fffff,taxi,other,H00024,25.5


### A5R. H3 derivable rico

In [34]:
df_a5r = make_h3_derivable(
    filas=15,
    seed=42,
    base_fields=["mode", "purpose", "trip_weight", "origin_municipality", "destination_municipality"],
    extra_columns=["household_id", "travel_time_min"],
)
trips_a5r, report_a5r = import_trips_from_dataframe(
    df_a5r,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_a5r",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)
display(trips_a5r.data.head(10))

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,mode,purpose,trip_weight,origin_municipality,destination_municipality,household_id,travel_time_min,origin_h3_index,destination_h3_index
0,m00000,u0000,t00000,0,-70.675453,-33.451344,-70.688240,-33.476079,2026-03-06 18:03:00+00:00,2026-03-06 18:57:00+00:00,taxi,leisure,4.355,Las Condes,Puente Alto,H00003,54.0,88b2c55431fffff,88b2c555d7fffff
1,m00001,u0006,t00001,0,-70.702833,-33.518244,-70.721648,-33.498726,2026-03-08 03:06:00+00:00,2026-03-08 04:27:00+00:00,car,education,3.913,Quilicura,La Florida,H00006,81.0,88b2c54751fffff,88b2c555a3fffff
2,m00002,u0005,t00002,0,-70.695213,-33.379648,-70.720768,-33.357351,2026-03-02 18:03:00+00:00,2026-03-02 18:22:00+00:00,bicycle,home,3.738,Las Condes,La Florida,H00002,19.0,88b2c55751fffff,88b2c5cda7fffff
3,m00003,u0003,t00003,0,-70.606769,-33.387777,-70.617775,-33.371482,2026-03-03 04:27:00+00:00,2026-03-03 05:36:00+00:00,car,home,2.444,Maipú,Pudahuel,H00007,69.0,88b2c51927fffff,88b2c55653fffff
4,m00004,u0003,t00004,0,-70.623456,-33.444718,-70.563658,-33.464683,2026-03-04 02:37:00+00:00,2026-03-04 03:40:00+00:00,scooter,errand,3.323,Vitacura,Santiago,H00002,63.0,88b2c554cbfffff,88b2c50b03fffff
5,m00005,u0006,t00005,0,-70.618727,-33.359821,-70.653360,-33.352856,2026-03-08 04:46:00+00:00,2026-03-08 06:21:00+00:00,motorcycle,other,3.128,Estación Central,Providencia,H00004,95.0,88b2c51965fffff,88b2c5564dfffff
6,m00006,u0000,t00006,0,-70.616918,-33.412599,-70.578187,-33.409099,2026-03-07 05:22:00+00:00,2026-03-07 07:21:00+00:00,bus,work,3.424,Recoleta,Ñuñoa,H00004,119.0,88b2c5568dfffff,88b2c556dbfffff
7,m00007,u0005,t00007,0,-70.445835,-33.518743,-70.513150,-33.512183,2026-03-01 11:49:00+00:00,2026-03-01 13:10:00+00:00,ride_hailing,errand,0.880,La Florida,La Florida,H00006,81.0,88b2c50f67fffff,88b2c50817fffff
8,m00008,u0001,t00008,0,-70.700642,-33.420500,-70.714037,-33.394357,2026-03-02 21:18:00+00:00,2026-03-02 22:10:00+00:00,ride_hailing,leisure,2.371,Maipú,Independencia,H00003,52.0,88b2c55461fffff,88b2c55703fffff
9,m00009,u0000,t00009,0,-70.711224,-33.526711,-70.704714,-33.520003,2026-03-07 00:02:00+00:00,2026-03-07 00:53:00+00:00,motorcycle,work,0.687,Recoleta,La Florida,H00002,51.0,88b2c54757fffff,88b2c54751fffff


## Sección 3. Errores estructurales

En esta sección se prueban casos en que `import_trips_from_dataframe(...)`
no debe completar el import y debe detenerse con excepción.

Los errores estructurales cubiertos aquí son:

- B1. Schema inválido
- B2. Missing required no derivable
- B3. `field_correspondence` inválido
- B4. `strict_domains=True` con extensión bloqueada
- B5. `movement_id` duplicado
- B6. H3 requerido no materializable

### B1. Schema inválido

In [35]:
df_b1 = make_happy_path_minimal(
    filas=6,
    seed=42,
)

INVALID_SCHEMA_B1 = TripSchema(
    version="",
    fields={
        "movement_id": FieldSpec("movement_id", "string", required=True),
        "user_id": FieldSpec("user_id", "string", required=True),
    },
    required=["movement_id", "user_id"],
)

In [36]:
try:
    import_trips_from_dataframe(
        df_b1,
        schema=INVALID_SCHEMA_B1,
        source_name="synthetic_b1",
        options=ImportOptions(),
        provenance=BASE_PROVENANCE,
        h3_resolution=8,
    )
    raise AssertionError("Se esperaba SchemaError y el import no falló.")
except SchemaError as exc:
    assert exc.code == "SCH.TRIP_SCHEMA.INVALID_VERSION"
    assert exc.issue is not None
    assert exc.issue.code == "SCH.TRIP_SCHEMA.INVALID_VERSION"
    assert exc.details is not None
    assert exc.details["schema_version"] == ""
    assert "expected" in exc.details

    assert exc.issues is not None
    assert isinstance(exc.issues, tuple)
    assert len(exc.issues) >= 1
    assert exc.issues[-1].code == "SCH.TRIP_SCHEMA.INVALID_VERSION"

### B2. Missing required no derivable

In [37]:
df_b2 = make_missing_required(
    filas=6,
    seed=42,
    missing_fields=["user_id"],
)

In [38]:
try:
    import_trips_from_dataframe(
        df_b2,
        schema=BASE_TRIP_SCHEMA,
        source_name="synthetic_b2",
        options=ImportOptions(),
        provenance=BASE_PROVENANCE,
        h3_resolution=8,
    )
    raise AssertionError("Se esperaba PylondrinaImportError y el import no falló.")
except PylondrinaImportError as exc:
    assert exc.code == "IMP.INPUT.MISSING_REQUIRED_FIELD"
    assert exc.issue is not None
    assert exc.issue.code == "IMP.INPUT.MISSING_REQUIRED_FIELD"

    assert exc.details is not None
    assert "missing_required" in exc.details
    assert "user_id" in exc.details["missing_required"]
    assert "required" in exc.details
    assert "source_columns" in exc.details

    assert exc.issues is not None
    assert isinstance(exc.issues, tuple)
    assert len(exc.issues) >= 1
    assert exc.issues[-1].code == "IMP.INPUT.MISSING_REQUIRED_FIELD"

### B3. field_correspondence inválido

In [39]:
df_b3 = make_happy_path_minimal(
    filas=6,
    seed=42,
)

field_corr_b3 = {
    "campo_que_no_existe_en_schema": "col_fuente_x",
}

In [40]:
try:
    import_trips_from_dataframe(
        df_b3,
        schema=BASE_TRIP_SCHEMA,
        source_name="synthetic_b3",
        options=ImportOptions(),
        field_correspondence=field_corr_b3,
        provenance=BASE_PROVENANCE,
        h3_resolution=8,
    )
    raise AssertionError("Se esperaba PylondrinaImportError y el import no falló.")
except PylondrinaImportError as exc:
    assert exc.code == "MAP.FIELDS.UNKNOWN_CANONICAL_FIELD"
    assert exc.issue is not None
    assert exc.issue.code == "MAP.FIELDS.UNKNOWN_CANONICAL_FIELD"

    assert exc.details is not None
    assert exc.details["field"] == "campo_que_no_existe_en_schema"
    assert "schema_fields_sample" in exc.details
    assert "schema_fields_total" in exc.details

    assert exc.issues is not None
    assert isinstance(exc.issues, tuple)
    assert len(exc.issues) >= 1
    assert exc.issues[-1].code == "MAP.FIELDS.UNKNOWN_CANONICAL_FIELD"

### B4. strict_domains=True con extensión bloqueada

In [41]:
field_corr_b4 = {
    "mode": "modo_fuente",
}

df_b4 = make_happy_path_minimal(
    filas=6,
    seed=42,
    base_fields=["mode"],
    field_correspondence=field_corr_b4,
)

df_b4["modo_fuente"] = ["PATIN", "PATIN", "PATIN", "PATIN", "PATIN", "PATIN"]

# "skateboard" no está en el dominio base de mode de tu BASE_TRIP_SCHEMA.
value_corr_b4 = {
    "mode": {
        "PATIN": "skateboard"
    }
}

In [42]:
try:
    import_trips_from_dataframe(
        df_b4,
        schema=BASE_TRIP_SCHEMA,
        source_name="synthetic_b4",
        options=ImportOptions(strict_domains=True),
        field_correspondence=field_corr_b4,
        value_correspondence=value_corr_b4,
        provenance=BASE_PROVENANCE,
        h3_resolution=8,
    )
    raise AssertionError("Se esperaba PylondrinaImportError y el import no falló.")
except PylondrinaImportError as exc:
    assert exc.code == "DOM.POLICY.MAPPING_REQUIRES_EXTENSION_BLOCKED"
    assert exc.issue is not None
    assert exc.issue.code == "DOM.POLICY.MAPPING_REQUIRES_EXTENSION_BLOCKED"

    assert exc.details is not None
    assert exc.details["field"] == "mode"
    assert exc.details["strict_domains"] is True
    assert exc.details["domain_extendable"] is True
    assert "unmapped_examples" in exc.details
    assert "skateboard" in exc.details["unmapped_examples"]

    assert exc.issues is not None
    assert isinstance(exc.issues, tuple)
    assert len(exc.issues) >= 1
    assert exc.issues[-1].code == "DOM.POLICY.MAPPING_REQUIRES_EXTENSION_BLOCKED"

### B5. movement_id duplicado

In [43]:
df_b5 = make_duplicate_movement_id(
    filas=8,
    seed=42,
    full_rows=False,
)

In [44]:
try:
    import_trips_from_dataframe(
        df_b5,
        schema=BASE_TRIP_SCHEMA,
        source_name="synthetic_b5",
        options=ImportOptions(),
        provenance=BASE_PROVENANCE,
        h3_resolution=8,
    )
    raise AssertionError("Se esperaba PylondrinaImportError y el import no falló.")
except PylondrinaImportError as exc:
    assert exc.code == "IMP.ID.MOVEMENT_ID_DUPLICATE"
    assert exc.issue is not None
    assert exc.issue.code == "IMP.ID.MOVEMENT_ID_DUPLICATE"

    assert exc.details is not None
    assert "duplicate_count" in exc.details
    assert exc.details["duplicate_count"] >= 2
    assert "duplicate_examples" in exc.details
    assert len(exc.details["duplicate_examples"]) >= 1

    assert exc.issues is not None
    assert isinstance(exc.issues, tuple)
    assert len(exc.issues) >= 1
    assert exc.issues[-1].code == "IMP.ID.MOVEMENT_ID_DUPLICATE"

### B6. H3 requerido no materializable

In [45]:
H3_REQUIRED_NO_COORDS_SCHEMA = TripSchema(
    version="1.1",
    fields={
        "movement_id": FieldSpec("movement_id", "string", required=True),
        "user_id": FieldSpec("user_id", "string", required=True),
        "trip_id": FieldSpec("trip_id", "string", required=True),
        "movement_seq": FieldSpec("movement_seq", "int", required=True),

        "origin_latitude": FieldSpec("origin_latitude", "float", required=False),
        "origin_longitude": FieldSpec("origin_longitude", "float", required=False),
        "destination_latitude": FieldSpec("destination_latitude", "float", required=False),
        "destination_longitude": FieldSpec("destination_longitude", "float", required=False),

        "origin_h3_index": FieldSpec("origin_h3_index", "string", required=True),
        "destination_h3_index": FieldSpec("destination_h3_index", "string", required=True),

        "origin_time_utc": FieldSpec("origin_time_utc", "datetime", required=False),
        "destination_time_utc": FieldSpec("destination_time_utc", "datetime", required=False),
    },
    required=[
        "movement_id",
        "user_id",
        "trip_id",
        "movement_seq",
        "origin_h3_index",
        "destination_h3_index",
    ],
)

In [46]:
df_b6 = generate_synthetic_trip_dataframe(
    filas=6,
    seed=42,
    tier_temporal="tier_1",
    tier1_datetime_format="utc_string_z",
    coord_format="numeric",
    h3_mode="omitted_derivable",
    trip_structure="independent",
    omit_required_fields=[
        "origin_h3_index",
        "destination_h3_index",
        "origin_latitude",
        "origin_longitude",
        "destination_latitude",
        "destination_longitude",
    ],
)

In [47]:
try:
    import_trips_from_dataframe(
        df_b6,
        schema=H3_REQUIRED_NO_COORDS_SCHEMA,
        source_name="synthetic_b6",
        options=ImportOptions(),
        provenance=BASE_PROVENANCE,
        h3_resolution=8,
    )
    raise AssertionError("Se esperaba PylondrinaImportError y el import no falló.")
except PylondrinaImportError as exc:
    assert exc.code == "IMP.H3.REQUIRED_FIELDS_UNAVAILABLE"
    assert exc.issue is not None
    assert exc.issue.code == "IMP.H3.REQUIRED_FIELDS_UNAVAILABLE"

    assert exc.details is not None
    assert "missing_pairs" in exc.details
    assert "required_h3_fields" in exc.details
    assert "origin_h3_index" in exc.details["required_h3_fields"]
    assert "destination_h3_index" in exc.details["required_h3_fields"]

    assert exc.issues is not None
    assert isinstance(exc.issues, tuple)
    assert len(exc.issues) >= 1
    assert exc.issues[-1].code == "IMP.H3.REQUIRED_FIELDS_UNAVAILABLE"

## Sección 4. Calidad no fatal y políticas

En esta sección se prueban casos donde `import_trips_from_dataframe(...)`
debe completar el import, pero dejando evidencia de calidad parcial,
políticas aplicadas y trazabilidad explícita en metadata y report.

Subbloques:

- C1. Dominio no extendable -> `unknown`
- C2. Dominio extendable -> `domains_effective` extendido
- C3. Tier 2 con HH:MM válidos e inválidos
- C4. Tier 3 sin tiempo OD
- C5. Datetime Tier 1 parcial
- C6. Coordenadas parciales
- C7. H3 parcial

Objeto nuevo

In [48]:
COORD_PARTIAL_NO_H3_SCHEMA = TripSchema(
    version="1.1",
    fields={
        "movement_id": FieldSpec("movement_id", "string", required=True),
        "user_id": FieldSpec("user_id", "string", required=True),
        "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
        "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
        "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
        "destination_latitude": FieldSpec("destination_latitude", "float", required=True),
        "origin_time_utc": FieldSpec("origin_time_utc", "datetime", required=False),
        "destination_time_utc": FieldSpec("destination_time_utc", "datetime", required=False),
        "trip_id": FieldSpec("trip_id", "string", required=True),
        "movement_seq": FieldSpec("movement_seq", "int", required=True),
        "mode": BASE_TRIP_SCHEMA.fields["mode"],
        "purpose": BASE_TRIP_SCHEMA.fields["purpose"],
    },
    required=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "trip_id",
        "movement_seq",
    ],
)

### C1. Dominio no extendable -> unknown

In [49]:
df_c1 = make_extended_domains(
    filas=10,
    seed=42,
    include_noise=False,
    extra_value_domains={
        "mode": ["canon", "submodes"],
    },
    base_fields=["mode", "purpose"],
)

In [50]:
trips_c1, report_c1 = import_trips_from_dataframe(
    df_c1,
    schema=NON_EXTENDABLE_DOMAIN_SCHEMA,
    source_name="synthetic_c1",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [51]:
assert isinstance(trips_c1, TripDataset)
assert isinstance(report_c1, ImportReport)
assert report_c1.ok is True

# issues
codes_c1 = [iss.code for iss in report_c1.issues]
assert "DOM.POLICY.FIELD_NOT_EXTENDABLE" in codes_c1

# contenido
assert "mode" in trips_c1.data.columns
assert "unknown" in set(trips_c1.data["mode"].dropna().astype(str).unique())

# domains_effective
mode_dom_c1 = trips_c1.metadata["domains_effective"]["mode"]
assert mode_dom_c1["extendable"] is False
assert mode_dom_c1["unknown_value"] == "unknown"
assert len(mode_dom_c1["unknown_values"]) >= 1
assert "unknown" in mode_dom_c1["values"]

# schema_effective / overrides
mode_overrides_c1 = trips_c1.metadata["schema_effective"]["overrides"].get("mode", {})
assert "reasons" in mode_overrides_c1
assert "out_of_domain_mapped_to_unknown" in mode_overrides_c1["reasons"]

# report profundo
assert isinstance(report_c1.issues, list)
assert report_c1.metadata["dataset_id"] == trips_c1.metadata["dataset_id"]
assert report_c1.metadata["summary"]["rows_out"] == len(trips_c1.data)

# evento
event_c1 = trips_c1.metadata["events"][-1]
assert event_c1["op"] == "import_trips"
assert "issues_summary" in event_c1
assert "counts" in event_c1["issues_summary"]
assert "by_code" in event_c1["issues_summary"]
assert event_c1["issues_summary"]["by_code"]["DOM.POLICY.FIELD_NOT_EXTENDABLE"] >= 1

In [52]:
display(df_c1.head(6))
display(trips_c1.data.head(6))

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,mode,purpose
0,m00000,u0000,t00000,0,-70.745929,-33.554174,-70.700770,-33.564738,2026-03-07T12:52:00-03:00,2026-03-07T14:28:00-03:00,88b2c54549fffff,88b2c54463fffff,Auto Acompañante,education
1,m00001,u0003,t00001,0,-70.623125,-33.439773,-70.627683,-33.423803,2026-03-03T06:26:00-03:00,2026-03-03T08:01:00-03:00,88b2c556b5fffff,88b2c556a9fffff,Bus troncal,transfer
2,m00002,u0003,t00002,0,-70.755888,-33.475299,-70.789495,-33.464336,2026-03-02T22:14:00-03:00,2026-03-02T23:48:00-03:00,88b2c542ddfffff,88b2c54211fffff,walk,home
3,m00003,u0002,t00003,0,-70.572155,-33.451344,-70.605134,-33.438962,2026-03-06T00:39:00-03:00,2026-03-06T02:00:00-03:00,88b2c50b0dfffff,88b2c556bbfffff,motorcycle,transfer
4,m00004,u0002,t00004,0,-70.664993,-33.518244,-70.638969,-33.505319,2026-03-05T16:58:00-03:00,2026-03-05T17:57:00-03:00,88b2c54601fffff,88b2c54651fffff,ride_hailing,education
5,m00005,u0004,t00005,0,-70.678486,-33.379648,-70.648756,-33.315399,2026-03-02T05:28:00-03:00,2026-03-02T06:54:00-03:00,88b2c55667fffff,88b2c5ccedfffff,scooter,other


,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,mode,purpose
0,m00000,u0000,t00000,0,-70.745929,-33.554174,-70.700770,-33.564738,2026-03-07 15:52:00+00:00,2026-03-07 17:28:00+00:00,88b2c54549fffff,88b2c54463fffff,unknown,education
1,m00001,u0003,t00001,0,-70.623125,-33.439773,-70.627683,-33.423803,2026-03-03 09:26:00+00:00,2026-03-03 11:01:00+00:00,88b2c556b5fffff,88b2c556a9fffff,unknown,transfer
2,m00002,u0003,t00002,0,-70.755888,-33.475299,-70.789495,-33.464336,2026-03-03 01:14:00+00:00,2026-03-03 02:48:00+00:00,88b2c542ddfffff,88b2c54211fffff,walk,home
3,m00003,u0002,t00003,0,-70.572155,-33.451344,-70.605134,-33.438962,2026-03-06 03:39:00+00:00,2026-03-06 05:00:00+00:00,88b2c50b0dfffff,88b2c556bbfffff,motorcycle,transfer
4,m00004,u0002,t00004,0,-70.664993,-33.518244,-70.638969,-33.505319,2026-03-05 19:58:00+00:00,2026-03-05 20:57:00+00:00,88b2c54601fffff,88b2c54651fffff,ride_hailing,education
5,m00005,u0004,t00005,0,-70.678486,-33.379648,-70.648756,-33.315399,2026-03-02 08:28:00+00:00,2026-03-02 09:54:00+00:00,88b2c55667fffff,88b2c5ccedfffff,scooter,other


### C2. Dominio extendable -> domains_effective extendido

In [53]:
df_c2 = make_extended_domains(
    filas=10,
    seed=42,
    include_noise=False,
    extra_value_domains={
        "mode": ["canon", "submodes"],
        "purpose": ["canon", "finos"],
    },
)

In [54]:
trips_c2, report_c2 = import_trips_from_dataframe(
    df_c2,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_c2",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [55]:
assert report_c2.ok is True

codes_c2 = [iss.code for iss in report_c2.issues]
assert "DOM.EXTENSION.APPLIED" in codes_c2

assert isinstance(trips_c2.metadata["domains_extended"], list)
assert len(trips_c2.metadata["domains_extended"]) >= 1

# revisar al menos un campo extendido
extended_fields_c2 = set(trips_c2.metadata["domains_extended"])
assert len(extended_fields_c2) >= 1

some_field_c2 = next(iter(extended_fields_c2))
dom_eff_c2 = trips_c2.metadata["domains_effective"][some_field_c2]

assert dom_eff_c2["extendable"] is True
assert dom_eff_c2["extended"] is True
assert dom_eff_c2["n_added"] >= 1
assert len(dom_eff_c2["added_values"]) >= 1
assert dom_eff_c2["strict_applied"] is False

# schema_effective profundo
overrides_c2 = trips_c2.metadata["schema_effective"]["overrides"][some_field_c2]
assert "reasons" in overrides_c2
assert "domain_extended" in overrides_c2["reasons"]
assert "added_values" in overrides_c2
assert len(overrides_c2["added_values"]) >= 1

# report profundo
assert report_c2.metadata["metadata"]["domains_extended"] == trips_c2.metadata["domains_extended"]
assert report_c2.metadata["metadata"]["dataset_id"] == trips_c2.metadata["dataset_id"]

### C3. Tier 2 con HH:MM válidos e inválidos

In [56]:
df_c3 = make_tier2_mixed_invalid(
    filas=12,
    seed=42,
    mostly_invalid=False,
)

In [57]:
trips_c3, report_c3 = import_trips_from_dataframe(
    df_c3,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_c3",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [58]:
assert report_c3.ok is True

codes_c3 = [iss.code for iss in report_c3.issues]
assert "IMP.TEMPORAL.TIER_LIMITED" in codes_c3
assert "IMP.TYPE.COERCE_PARTIAL" in codes_c3

# tabular
assert "origin_time_local_hhmm" in trips_c3.data.columns
assert "destination_time_local_hhmm" in trips_c3.data.columns
assert trips_c3.data["origin_time_local_hhmm"].isna().sum() + trips_c3.data["destination_time_local_hhmm"].isna().sum() >= 1

# temporal metadata
temporal_c3 = trips_c3.metadata["temporal"]
assert temporal_c3["tier"] == "tier_2"
assert "normalization" in temporal_c3
assert "origin_time_local_hhmm" in temporal_c3["normalization"]
assert "destination_time_local_hhmm" in temporal_c3["normalization"]

# stats profundas
stats_o_c3 = temporal_c3["normalization"]["origin_time_local_hhmm"]
stats_d_c3 = temporal_c3["normalization"]["destination_time_local_hhmm"]
assert "n_invalid" in stats_o_c3
assert "n_total" in stats_o_c3
assert "n_invalid" in stats_d_c3
assert "n_total" in stats_d_c3

# report profundo
assert report_c3.metadata["metadata"]["temporal"]["tier"] == "tier_2"
assert report_c3.metadata["summary"]["rows_in"] == 12
assert report_c3.metadata["summary"]["rows_out"] == 12

### C4. Tier 3 sin tiempo OD

In [59]:
df_c4 = generate_synthetic_trip_dataframe(
    filas=10,
    seed=42,
    tier_temporal="tier_3",
    coord_format="numeric",
    h3_mode="provided_valid",
    trip_structure="independent",
    base_fields=["mode", "purpose"],
)

In [60]:
trips_c4, report_c4 = import_trips_from_dataframe(
    df_c4,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_c4",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [61]:
assert report_c4.ok is True

codes_c4 = [iss.code for iss in report_c4.issues]
assert "IMP.TEMPORAL.TIER_LIMITED" in codes_c4

temporal_c4 = trips_c4.metadata["temporal"]
assert temporal_c4["tier"] == "tier_3"
assert temporal_c4["fields_present"] == []

# en tier_3 no debería agregarse normalization temporal
assert "normalization" not in temporal_c4

# reporte y metadata
assert report_c4.metadata["metadata"]["temporal"]["tier"] == "tier_3"
assert trips_c4.metadata["is_validated"] is False

### C5. Datetime Tier 1 parcial

In [62]:
df_c5 = generate_synthetic_trip_dataframe(
    filas=12,
    seed=42,
    tier_temporal="tier_1",
    tier1_datetime_format="mixed_with_invalids",
    coord_format="numeric",
    h3_mode="provided_valid",
    trip_structure="independent",
    base_fields=["mode", "purpose"],
)

In [63]:
display(df_c5)

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,mode,purpose
0,m00000,u0000,t00000,0,-70.572155,-33.439773,-70.598775,-33.375523,2026/99/99 25:61:00,2026-03-05 05:03:00+00:00,88b2c50b41fffff,88b2c5192bfffff,bicycle,other
1,m00001,u0004,t00001,0,-70.664993,-33.475299,-70.655706,-33.487492,2026-03-04T11:05:00Z,2026-03-04T12:15:00Z,88b2c554adfffff,88b2c554a3fffff,walk,home
2,m00002,u0003,t00002,0,-70.678486,-33.451344,-70.673819,-33.466711,2026-03-04 18:56:00,2026-03-04 19:10:00,88b2c55431fffff,88b2c555dbfffff,metro,shopping
3,m00003,u0002,t00003,0,-70.728093,-33.518244,-70.719345,-33.542657,2026-03-05 05:32:00,not-a-date,88b2c54747fffff,88b2c54715fffff,bus,work
4,m00004,u0002,t00004,0,-70.537746,-33.379648,-70.502889,-33.361169,2026-03-01 12:15:00,2026-03-01 13:26:00,88b2c519c9fffff,88b2c5181dfffff,bus,shopping
5,m00005,u0005,t00005,0,-70.675453,-33.387777,-70.666509,-33.353907,2026-03-02T05:29:00-03:00,abc,88b2c55621fffff,88b2c5cd93fffff,taxi,other
6,m00006,u0000,t00006,0,-70.702833,-33.444718,-70.675676,-33.448136,2026-03-02T23:14:00-03:00,2026-03-03T00:24:00-03:00,88b2c55427fffff,88b2c55431fffff,bus,shopping
7,m00007,u0004,t00007,0,-70.695213,-33.359821,-70.692510,-33.385025,2026-03-02 01:14:00,2026-03-02 02:22:00,88b2c5cdb1fffff,88b2c55753fffff,bicycle,other
8,m00008,u0001,t00008,0,-70.606769,-33.412599,-70.595204,-33.437334,2026-03-04 07:51:00,2026-03-04 08:06:00,88b2c55689fffff,88b2c55697fffff,train,errand
9,m00009,u0000,t00009,0,-70.623456,-33.518743,-70.598204,-33.499226,123456,2026-03-05 23:26:00,88b2c50921fffff,88b2c5090dfffff,taxi,leisure


In [64]:
trips_c5, report_c5 = import_trips_from_dataframe(
    df_c5,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_c5",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

D:\Memoria\repos\pylondrina\src\pylondrina\importing.py:1925: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  parsed = pd.to_datetime(tmp, format="mixed", errors="coerce")
D:\Memoria\repos\pylondrina\src\pylondrina\importing.py:1925: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  parsed = pd.to_datetime(tmp, format="mixed", errors="coerce")


In [65]:
display(trips_c5.data)

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,mode,purpose
0,m00000,u0000,t00000,0,-70.572155,-33.439773,-70.598775,-33.375523,NaT,2026-03-05 05:03:00+00:00,88b2c50b41fffff,88b2c5192bfffff,bicycle,other
1,m00001,u0004,t00001,0,-70.664993,-33.475299,-70.655706,-33.487492,2026-03-04 11:05:00+00:00,2026-03-04 12:15:00+00:00,88b2c554adfffff,88b2c554a3fffff,walk,home
2,m00002,u0003,t00002,0,-70.678486,-33.451344,-70.673819,-33.466711,2026-03-04 18:56:00+00:00,2026-03-04 19:10:00+00:00,88b2c55431fffff,88b2c555dbfffff,metro,shopping
3,m00003,u0002,t00003,0,-70.728093,-33.518244,-70.719345,-33.542657,2026-03-05 05:32:00+00:00,NaT,88b2c54747fffff,88b2c54715fffff,bus,work
4,m00004,u0002,t00004,0,-70.537746,-33.379648,-70.502889,-33.361169,2026-03-01 12:15:00+00:00,2026-03-01 13:26:00+00:00,88b2c519c9fffff,88b2c5181dfffff,bus,shopping
5,m00005,u0005,t00005,0,-70.675453,-33.387777,-70.666509,-33.353907,2026-03-02 08:29:00+00:00,NaT,88b2c55621fffff,88b2c5cd93fffff,taxi,other
6,m00006,u0000,t00006,0,-70.702833,-33.444718,-70.675676,-33.448136,2026-03-03 02:14:00+00:00,2026-03-03 03:24:00+00:00,88b2c55427fffff,88b2c55431fffff,bus,shopping
7,m00007,u0004,t00007,0,-70.695213,-33.359821,-70.692510,-33.385025,2026-03-02 01:14:00+00:00,2026-03-02 02:22:00+00:00,88b2c5cdb1fffff,88b2c55753fffff,bicycle,other
8,m00008,u0001,t00008,0,-70.606769,-33.412599,-70.595204,-33.437334,2026-03-04 07:51:00+00:00,2026-03-04 08:06:00+00:00,88b2c55689fffff,88b2c55697fffff,train,errand
9,m00009,u0000,t00009,0,-70.623456,-33.518743,-70.598204,-33.499226,NaT,2026-03-05 23:26:00+00:00,88b2c50921fffff,88b2c5090dfffff,taxi,leisure


In [66]:
display(report_c5.issues)

[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'day_type' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='day_type', source_field=None, row_count=None, details={'field': 'day_type', 'source_columns': ['movement_id', 'user_id', 'trip_id', 'movement_seq', 'origin_longitude', 'origin_latitude', 'destination_longitude', 'destination_latitude', 'origin_time_utc', 'destination_time_utc', 'origin_h3_index', 'destination_h3_index', 'mode', 'purpose'], 'field_correspondence_used': False, 'action': 'omit_optional'}),
 Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'time_period' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='time_period', source_field=None, row_count=None, details={'field': 'time_period', 'source_columns': ['movement_id', 'user_id', 'trip_id', 'movement_seq', 'origin_longitude', 'origin_latitude', 'destination_longitude'

In [67]:
assert report_c5.ok is True

codes_c5 = [iss.code for iss in report_c5.issues]
assert "IMP.TYPE.COERCE_PARTIAL" in codes_c5

# tabular
assert "origin_time_utc" in trips_c5.data.columns
assert "destination_time_utc" in trips_c5.data.columns
assert trips_c5.data["origin_time_utc"].isna().sum() + trips_c5.data["destination_time_utc"].isna().sum() >= 1

# temporal metadata
temporal_c5 = trips_c5.metadata["temporal"]
assert temporal_c5["tier"] == "tier_1"
assert "normalization" in temporal_c5
assert "origin_time_utc" in temporal_c5["normalization"]
assert "destination_time_utc" in temporal_c5["normalization"]

# info profunda de normalización
norm_o_c5 = temporal_c5["normalization"]["origin_time_utc"]
norm_d_c5 = temporal_c5["normalization"]["destination_time_utc"]
assert "status" in norm_o_c5
assert "status" in norm_d_c5
assert "n_nat" in norm_o_c5
assert "n_nat" in norm_d_c5

# report profundo
assert isinstance(report_c5.metadata["metadata"]["temporal"]["normalization"], dict)
assert report_c5.metadata["dataset_id"] == trips_c5.metadata["dataset_id"]

### C6. Coordenadas parciales

foco principal en coerción parcial de coordenadas

In [68]:
df_c6 = generate_synthetic_trip_dataframe(
    filas=12,
    seed=42,
    tier_temporal="tier_1",
    tier1_datetime_format="utc_string_z",
    coord_format="numeric",
    h3_mode="provided_valid",
    trip_structure="independent",
    base_fields=["mode", "purpose"],
    type_corruption={
        "origin_latitude": {"mode": "non_numeric_text", "ratio": 0.25},
        "destination_longitude": {"mode": "non_numeric_text", "ratio": 0.30},
    },
)

In [69]:
display(df_c6.head())

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,mode,purpose
0,m00000,u0000,t00000,0,-70.572155,-33.439773,-70.598775,-33.375523,2026-03-05T03:20:00Z,2026-03-05T05:03:00Z,88b2c50b41fffff,88b2c5192bfffff,train,leisure
1,m00001,u0004,t00001,0,-70.664993,no_num,xx,-33.487492,2026-03-04T11:05:00Z,2026-03-04T12:15:00Z,88b2c554adfffff,88b2c554a3fffff,walk,leisure
2,m00002,u0003,t00002,0,-70.678486,-33.451344,-70.673819,-33.466711,2026-03-04T18:56:00Z,2026-03-04T19:10:00Z,88b2c55431fffff,88b2c555dbfffff,train,leisure
3,m00003,u0002,t00003,0,-70.728093,-33.518244,no_num,-33.542657,2026-03-05T05:32:00Z,2026-03-05T07:04:00Z,88b2c54747fffff,88b2c54715fffff,motorcycle,errand
4,m00004,u0002,t00004,0,-70.537746,texto,-70.502889,-33.361169,2026-03-01T12:15:00Z,2026-03-01T13:26:00Z,88b2c519c9fffff,88b2c5181dfffff,other,leisure


In [70]:
trips_c6, report_c6 = import_trips_from_dataframe(
    df_c6,
    schema=COORD_PARTIAL_NO_H3_SCHEMA,
    source_name="synthetic_c6",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)
display(trips_c6.data.head())

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,mode,purpose
0,m00000,u0000,t00000,0,-70.572155,-33.439773,-70.598775,-33.375523,2026-03-05 03:20:00+00:00,2026-03-05 05:03:00+00:00,88b2c50b41fffff,88b2c5192bfffff,train,leisure
1,m00001,u0004,t00001,0,-70.664993,NaN,NaN,-33.487492,2026-03-04 11:05:00+00:00,2026-03-04 12:15:00+00:00,<NA>,<NA>,walk,leisure
2,m00002,u0003,t00002,0,-70.678486,-33.451344,-70.673819,-33.466711,2026-03-04 18:56:00+00:00,2026-03-04 19:10:00+00:00,88b2c55431fffff,88b2c555dbfffff,train,leisure
3,m00003,u0002,t00003,0,-70.728093,-33.518244,NaN,-33.542657,2026-03-05 05:32:00+00:00,2026-03-05 07:04:00+00:00,88b2c54747fffff,<NA>,motorcycle,errand
4,m00004,u0002,t00004,0,-70.537746,NaN,-70.502889,-33.361169,2026-03-01 12:15:00+00:00,2026-03-01 13:26:00+00:00,<NA>,88b2c5181dfffff,other,leisure


In [71]:
assert report_c6.ok is True

codes_c6 = [iss.code for iss in report_c6.issues]
assert "IMP.TYPE.COERCE_PARTIAL" in codes_c6

# tabular
assert trips_c6.data["origin_latitude"].isna().sum() >= 1
assert trips_c6.data["destination_longitude"].isna().sum() >= 1

# El contrato conceptual del formato de datos Golondrina materializa H3 cuando detecta coordenadas,
# entonces es válido que aparezca metadata h3
assert "h3" in trips_c6.metadata
assert trips_c6.metadata["h3"]["resolution"] == 8

# si hubo filas con coordenadas incompletas o inválidas, H3 también puede quedar parcial
assert "origin_h3_index" in trips_c6.data.columns
assert "destination_h3_index" in trips_c6.data.columns

# no se exige que todo H3 sea nulo ni todo completo; solo que el pipeline siguió y dejó trazabilidad
event_c6 = trips_c6.metadata["events"][-1]
assert event_c6["issues_summary"]["by_code"]["IMP.TYPE.COERCE_PARTIAL"] >= 1

# metadata / report
assert report_c6.metadata["summary"]["rows_out"] == len(trips_c6.data)
assert report_c6.metadata["metadata"]["dataset_id"] == trips_c6.metadata["dataset_id"]

### C7. H3 parcial

foco principal en parcialidad H3 derivada

In [72]:
df_c7 = generate_synthetic_trip_dataframe(
    filas=12,
    seed=42,
    tier_temporal="tier_1",
    tier1_datetime_format="utc_string_z",
    coord_format="numeric",
    h3_mode="omitted_derivable",
    trip_structure="independent",
    base_fields=["mode", "purpose"],
    paired_missingness={
        "origin_incomplete": 0.20,
        "destination_incomplete": 0.25,
    },
    omit_required_fields=["origin_h3_index", "destination_h3_index"],
)

In [73]:
trips_c7, report_c7 = import_trips_from_dataframe(
    df_c7,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_c7",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [74]:
assert report_c7.ok is True

codes_c7 = [iss.code for iss in report_c7.issues]
assert "IMP.H3.PARTIAL_DERIVATION" in codes_c7

# tabular
assert "origin_h3_index" in trips_c7.data.columns
assert "destination_h3_index" in trips_c7.data.columns
assert trips_c7.data["origin_h3_index"].isna().sum() + trips_c7.data["destination_h3_index"].isna().sum() >= 1

# metadata h3
assert "h3" in trips_c7.metadata
h3_c7 = trips_c7.metadata["h3"]
assert h3_c7["resolution"] == 8
assert "source_fields" in h3_c7
assert "derived_fields" in h3_c7
assert "origin_h3_index" in h3_c7["derived_fields"] or "destination_h3_index" in h3_c7["derived_fields"]

# report profundo
assert report_c7.metadata["metadata"]["h3"]["resolution"] == 8

# evento
event_c7 = trips_c7.metadata["events"][-1]
assert event_c7["parameters"]["h3_resolution"] == 8
assert event_c7["issues_summary"]["by_code"]["IMP.H3.PARTIAL_DERIVATION"] >= 1

### C2R. Dominio extendable rico

In [75]:
df_c2r = make_extended_domains(
    filas=50,
    seed=42,
    include_noise=True,
    extra_value_domains={
        "mode": ["canon", "submodes"],
        "purpose": ["canon", "finos"],
        "time_period": ["canon", "eod_finos"],
    },
    base_fields=["mode", "purpose", "time_period", "user_gender", "trip_weight"],
    extra_columns=["household_id", "travel_time_min", "survey_date"],
)
trips_c2r, report_c2r = import_trips_from_dataframe(
    df_c2r,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_c2r",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)
assert report_c2r.ok is True

codes_c2r = [iss.code for iss in report_c2r.issues]
assert "DOM.EXTENSION.APPLIED" in codes_c2r

assert len(trips_c2r.data) == 50
assert len(trips_c2r.metadata["domains_extended"]) >= 1

# revisar profundidad report.metadata
assert report_c2r.metadata["summary"]["rows_in"] == 50
assert report_c2r.metadata["summary"]["rows_out"] == 50
assert report_c2r.metadata["metadata"]["dataset_id"] == trips_c2r.metadata["dataset_id"]
assert isinstance(report_c2r.metadata["metadata"]["domains_effective"], dict)

event_c2r = trips_c2r.metadata["events"][-1]
assert event_c2r["summary"]["input_rows"] == 50
assert event_c2r["summary"]["output_rows"] == 50

### C3R. Tier 2 mixto rico

In [76]:
df_c3r = make_tier2_mixed_invalid(
    filas=50,
    seed=42,
    mostly_invalid=True,
    base_fields=["mode", "purpose", "trip_weight"],
    extra_columns=["household_id", "travel_time_min", "survey_date"],
)

trips_c3r, report_c3r = import_trips_from_dataframe(
    df_c3r,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_c3r",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

assert report_c3r.ok is True

codes_c3r = [iss.code for iss in report_c3r.issues]
assert "IMP.TEMPORAL.TIER_LIMITED" in codes_c3r
assert "IMP.TYPE.COERCE_PARTIAL" in codes_c3r

assert len(trips_c3r.data) == 50
assert trips_c3r.metadata["temporal"]["tier"] == "tier_2"

norm_c3r = trips_c3r.metadata["temporal"]["normalization"]
assert norm_c3r["origin_time_local_hhmm"]["n_total"] == 50
assert norm_c3r["destination_time_local_hhmm"]["n_total"] == 50
assert norm_c3r["origin_time_local_hhmm"]["n_invalid"] + norm_c3r["destination_time_local_hhmm"]["n_invalid"] >= 1

assert report_c3r.metadata["metadata"]["temporal"]["tier"] == "tier_2"

### C7R. H3 parcial rico

In [77]:
df_c7r = generate_synthetic_trip_dataframe(
    filas=50,
    seed=42,
    tier_temporal="tier_1",
    tier1_datetime_format="offset_string",
    coord_format="numeric",
    h3_mode="omitted_derivable",
    trip_structure="independent",
    base_fields=["mode", "purpose", "trip_weight", "origin_municipality", "destination_municipality"],
    extra_columns=["household_id", "travel_time_min", "survey_date"],
    paired_missingness={
        "origin_incomplete": 0.18,
        "destination_incomplete": 0.22,
    },
    omit_required_fields=["origin_h3_index", "destination_h3_index"],
)

trips_c7r, report_c7r = import_trips_from_dataframe(
    df_c7r,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_c7r",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

assert report_c7r.ok is True

codes_c7r = [iss.code for iss in report_c7r.issues]
assert "IMP.H3.PARTIAL_DERIVATION" in codes_c7r

assert len(trips_c7r.data) == 50
assert trips_c7r.data["origin_h3_index"].isna().sum() + trips_c7r.data["destination_h3_index"].isna().sum() >= 1

assert trips_c7r.metadata["h3"]["resolution"] == 8
assert len(trips_c7r.metadata["h3"]["derived_fields"]) >= 1

event_c7r = trips_c7r.metadata["events"][-1]
assert event_c7r["summary"]["input_rows"] == 50
assert event_c7r["summary"]["output_rows"] == 50
assert event_c7r["issues_summary"]["by_code"]["IMP.H3.PARTIAL_DERIVATION"] >= 1

assert report_c7r.metadata["metadata"]["h3"]["resolution"] == 8

## Sección 5. Selección final y salida observable

En esta sección se prueba cómo `import_trips_from_dataframe(...)` decide
qué columnas sobreviven al resultado final, considerando:

- `selected_fields`
- `keep_extra_fields`

El foco aquí es verificar:

- columnas del dataset final,
- columnas eliminadas,
- `extra_fields_kept`,
- `fields_effective` y poda de `schema_effective`,
- consistencia entre `TripDataset.metadata`, `ImportReport.metadata` y el evento `import_trips`.

### Dataset base para la pasada controlada

In [78]:
df_d_base = make_happy_path_rich(
    filas=12,
    seed=42,
    base_fields=[
        "mode",
        "purpose",
        "day_type",
        "time_period",
        "user_gender",
        "trip_weight",
        "origin_municipality",
        "destination_municipality",
    ],
    extra_columns=[
        "household_id",
        "source_person_id",
        "stage_count",
        "activity_destination",
        "travel_time_min",
        "fare_amount",
    ],
)

### Conjuntos auxiliares para asserts

In [79]:
required_cols = set(BASE_TRIP_SCHEMA.required)

schema_optional_present_in_source = {
    "mode",
    "purpose",
    "day_type",
    "time_period",
    "user_gender",
    "trip_weight",
    "origin_municipality",
    "destination_municipality",
}

extra_cols_present_in_source = {
    "household_id",
    "source_person_id",
    "stage_count",
    "activity_destination",
    "travel_time_min",
    "fare_amount",
}

subset_selected = ["mode", "purpose", "trip_weight"]
expected_subset_schema_cols = required_cols | set(subset_selected)
expected_all_schema_cols = required_cols | schema_optional_present_in_source | set(["destination_time_utc", "origin_time_utc"])

### D1. selected_fields=None + keep_extra_fields=False

In [80]:
trips_d1, report_d1 = import_trips_from_dataframe(
    df_d_base,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_d1",
    options=ImportOptions(
        selected_fields=None,
        keep_extra_fields=False,
    ),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [81]:
assert report_d1.ok is True
assert isinstance(trips_d1, TripDataset)
assert isinstance(report_d1, ImportReport)

final_cols_d1 = set(trips_d1.data.columns)

# tabular: deben quedar todos los campos del schema presentes en la fuente
assert expected_all_schema_cols.issubset(final_cols_d1)

# tabular: no deben quedar extras
assert final_cols_d1.isdisjoint(extra_cols_present_in_source)

# operacional: se debe haber emitido issue por extras borradas
codes_d1 = [iss.code for iss in report_d1.issues]
assert "IMP.OPTIONS.EXTRA_FIELDS_DROPPED" in codes_d1

# metadata: extras efectivamente no quedaron
assert trips_d1.metadata["extra_fields_kept"] == []

# schema_effective podado al resultado final
fields_eff_d1 = set(trips_d1.metadata["schema_effective"]["fields_effective"])

assert sorted(fields_eff_d1) == sorted(expected_all_schema_cols)

# report profundo
assert report_d1.metadata["summary"]["rows_in"] == 12
assert report_d1.metadata["summary"]["rows_out"] == 12
assert report_d1.metadata["metadata"]["extra_fields_kept"] == []
assert set(report_d1.metadata["metadata"]["schema_effective"]["fields_effective"]) == expected_all_schema_cols

# evento profundo
event_d1 = trips_d1.metadata["events"][-1]
assert event_d1["op"] == "import_trips"
assert set(event_d1["summary"]["columns_deleted"]) >= extra_cols_present_in_source
assert event_d1["summary"]["output_rows"] == 12
assert event_d1["issues_summary"]["by_code"]["IMP.OPTIONS.EXTRA_FIELDS_DROPPED"] >= 1

### D2. selected_fields=None + keep_extra_fields=True

In [82]:
trips_d2, report_d2 = import_trips_from_dataframe(
    df_d_base,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_d2",
    options=ImportOptions(
        selected_fields=None,
        keep_extra_fields=True,
    ),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [83]:
assert report_d2.ok is True

final_cols_d2 = set(trips_d2.data.columns)

# tabular: se conservan todos los fields del schema presentes
assert expected_all_schema_cols.issubset(final_cols_d2)

# tabular: se conservan también las extras
assert extra_cols_present_in_source.issubset(final_cols_d2)

# operacional: no debería aparecer el issue de extras borradas
codes_d2 = [iss.code for iss in report_d2.issues]
assert "IMP.OPTIONS.EXTRA_FIELDS_DROPPED" not in codes_d2

# metadata: extras registradas
assert set(trips_d2.metadata["extra_fields_kept"]) == extra_cols_present_in_source

# schema_effective solo lista fields del schema, no extras
fields_eff_d2 = set(trips_d2.metadata["schema_effective"]["fields_effective"])
assert fields_eff_d2 == expected_all_schema_cols
assert fields_eff_d2.isdisjoint(extra_cols_present_in_source)

# report profundo
assert report_d2.metadata["metadata"]["dataset_id"] == trips_d2.metadata["dataset_id"]
assert set(report_d2.metadata["metadata"]["extra_fields_kept"]) == extra_cols_present_in_source
assert isinstance(report_d2.metadata["metadata"]["schema_effective"]["fields_effective"], list)

# evento profundo
event_d2 = trips_d2.metadata["events"][-1]
assert event_d2["summary"]["columns_deleted"] == []
assert event_d2["summary"]["output_rows"] == 12

### D3. selected_fields=subconjunto + keep_extra_fields=False

In [84]:
trips_d3, report_d3 = import_trips_from_dataframe(
    df_d_base,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_d3",
    options=ImportOptions(
        selected_fields=subset_selected,
        keep_extra_fields=False,
    ),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [85]:
assert report_d3.ok is True

final_cols_d3 = set(trips_d3.data.columns)

# tabular: deben quedar los required + subset seleccionado
assert expected_subset_schema_cols.issubset(final_cols_d3)

# tabular: no deben quedar otros opcionales del schema que no fueron seleccionados
dropped_optional_schema_d3 = schema_optional_present_in_source - set(subset_selected)
assert final_cols_d3.isdisjoint(dropped_optional_schema_d3)

# tabular: tampoco deben quedar extras
assert final_cols_d3.isdisjoint(extra_cols_present_in_source)

# operacional: issue por extras borradas
codes_d3 = [iss.code for iss in report_d3.issues]
assert "IMP.OPTIONS.EXTRA_FIELDS_DROPPED" in codes_d3

# metadata
assert trips_d3.metadata["extra_fields_kept"] == []

fields_eff_d3 = set(trips_d3.metadata["schema_effective"]["fields_effective"])
assert fields_eff_d3 == expected_subset_schema_cols

# report profundo
assert set(report_d3.metadata["metadata"]["schema_effective"]["fields_effective"]) == expected_subset_schema_cols
assert report_d3.parameters["selected_fields"] == subset_selected
assert report_d3.parameters["keep_extra_fields"] is False

# evento profundo
event_d3 = trips_d3.metadata["events"][-1]
assert set(event_d3["summary"]["columns_deleted"]) >= (extra_cols_present_in_source | dropped_optional_schema_d3)

### D4. selected_fields=subconjunto + keep_extra_fields=True

In [86]:
trips_d4, report_d4 = import_trips_from_dataframe(
    df_d_base,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_d4",
    options=ImportOptions(
        selected_fields=subset_selected,
        keep_extra_fields=True,
    ),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [87]:
assert report_d4.ok is True

final_cols_d4 = set(trips_d4.data.columns)

# tabular: quedan required + subset
assert expected_subset_schema_cols.issubset(final_cols_d4)

# tabular: opcionales del schema no seleccionados deben caer
dropped_optional_schema_d4 = schema_optional_present_in_source - set(subset_selected)
assert final_cols_d4.isdisjoint(dropped_optional_schema_d4)

# tabular: extras sí se conservan
assert extra_cols_present_in_source.issubset(final_cols_d4)

# operacional: como no se borran extras, no debería salir este issue
codes_d4 = [iss.code for iss in report_d4.issues]
assert "IMP.OPTIONS.EXTRA_FIELDS_DROPPED" not in codes_d4

# metadata
assert set(trips_d4.metadata["extra_fields_kept"]) == extra_cols_present_in_source

fields_eff_d4 = set(trips_d4.metadata["schema_effective"]["fields_effective"])
assert fields_eff_d4 == expected_subset_schema_cols

# report profundo
assert report_d4.parameters["selected_fields"] == subset_selected
assert report_d4.parameters["keep_extra_fields"] is True
assert set(report_d4.metadata["metadata"]["extra_fields_kept"]) == extra_cols_present_in_source
assert set(report_d4.metadata["metadata"]["schema_effective"]["fields_effective"]) == expected_subset_schema_cols

# evento profundo
event_d4 = trips_d4.metadata["events"][-1]
assert set(event_d4["summary"]["columns_deleted"]) >= dropped_optional_schema_d4
assert event_d4["summary"]["columns_deleted"] == list(event_d4["summary"]["columns_deleted"])
assert event_d4["summary"]["output_rows"] == 12

In [88]:
display(trips_d4.data.head())

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index,mode,purpose,trip_weight,household_id,source_person_id,stage_count,activity_destination,travel_time_min,fare_amount
0,m00000,u0000,t00000,0,-70.572155,-33.439773,-70.598775,-33.375523,88b2c50b41fffff,88b2c5192bfffff,train,leisure,0.965,H00003,P00004,1,salud,103.0,1200.0
1,m00001,u0004,t00001,0,-70.664993,-33.475299,-70.655706,-33.487492,88b2c554adfffff,88b2c554a3fffff,walk,leisure,3.144,H00000,P00004,3,estudio,70.0,1200.0
2,m00002,u0003,t00002,0,-70.678486,-33.451344,-70.673819,-33.466711,88b2c55431fffff,88b2c555dbfffff,train,leisure,1.268,H00001,P00005,3,salud,14.0,800.0
3,m00003,u0002,t00003,0,-70.728093,-33.518244,-70.719345,-33.542657,88b2c54747fffff,88b2c54715fffff,motorcycle,errand,4.663,H00003,P00004,2,compra,92.0,760.0
4,m00004,u0002,t00004,0,-70.537746,-33.379648,-70.502889,-33.361169,88b2c519c9fffff,88b2c5181dfffff,other,leisure,3.115,H00001,P00001,2,recreación,71.0,0.0


### Dataset "rico" común

In [89]:
df_d_rich = make_happy_path_rich(
    filas=50,
    seed=42,
    base_fields=[
        "mode",
        "purpose",
        "day_type",
        "time_period",
        "user_gender",
        "trip_weight",
        "origin_municipality",
        "destination_municipality",
    ],
    extra_columns=[
        "household_id",
        "source_person_id",
        "stage_count",
        "activity_destination",
        "travel_time_min",
        "fare_amount",
    ],
)

In [90]:
display(df_d_rich.head())

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,mode,purpose,day_type,time_period,user_gender,trip_weight,origin_municipality,destination_municipality,household_id,source_person_id,stage_count,activity_destination,travel_time_min,fare_amount
0,m00000,u0002,t00000,0,-70.590951,-33.478171,-70.584048,-33.467465,2026-03-06T11:48:00-03:00,2026-03-06T13:29:00-03:00,88b2c50949fffff,88b2c50b2bfffff,metro,leisure,holiday,morning,female,3.674,Quilicura,La Florida,H00011,P00019,2,estudio,101.0,760.0
1,m00001,u0019,t00001,0,-70.702725,-33.407415,-70.639562,-33.363516,2026-03-04T04:24:00-03:00,2026-03-04T06:02:00-03:00,88b2c55717fffff,88b2c55643fffff,car,leisure,weekday,evening,male,2.245,San Miguel,Vitacura,H00020,P00021,1,salud,98.0,1520.0
2,m00002,u0016,t00002,0,-70.644146,-33.420764,-70.637746,-33.456427,2026-03-01T07:05:00-03:00,2026-03-01T09:00:00-03:00,88b2c5545bfffff,88b2c554c7fffff,bus,other,holiday,midday,other,3.384,San Miguel,Providencia,H00005,P00022,3,compra,115.0,900.0
3,m00003,u0010,t00003,0,-70.597441,-33.416981,-70.602186,-33.436174,2026-03-08T00:19:00-03:00,2026-03-08T00:36:00-03:00,88b2c5568bfffff,88b2c556bbfffff,bus,transfer,holiday,midday,unknown,0.548,Providencia,Providencia,H00000,P00007,1,salud,17.0,800.0
4,m00004,u0010,t00004,0,-70.690935,-33.415534,-70.679502,-33.443332,2026-03-02T09:52:00-03:00,2026-03-02T11:38:00-03:00,88b2c55469fffff,88b2c5543dfffff,scooter,errand,weekday,midday,male,1.441,Santiago,San Miguel,H00001,P00017,2,salud,106.0,1520.0


### D2R. selected_fields=None + keep_extra_fields=True rico

In [91]:
trips_d2r, report_d2r = import_trips_from_dataframe(
    df_d_rich,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_d2r",
    options=ImportOptions(
        selected_fields=None,
        keep_extra_fields=True,
    ),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [92]:
assert report_d2r.ok is True
assert len(trips_d2r.data) == 50

final_cols_d2r = set(trips_d2r.data.columns)
assert expected_all_schema_cols.issubset(final_cols_d2r)
assert extra_cols_present_in_source.issubset(final_cols_d2r)

# metadata / report profundo
assert set(trips_d2r.metadata["extra_fields_kept"]) == extra_cols_present_in_source
assert set(report_d2r.metadata["metadata"]["extra_fields_kept"]) == extra_cols_present_in_source
assert report_d2r.metadata["summary"]["rows_in"] == 50
assert report_d2r.metadata["summary"]["rows_out"] == 50

event_d2r = trips_d2r.metadata["events"][-1]
assert event_d2r["summary"]["input_rows"] == 50
assert event_d2r["summary"]["output_rows"] == 50
assert event_d2r["summary"]["columns_deleted"] == []

In [93]:
display(trips_d2r.data.head())

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,mode,purpose,day_type,time_period,user_gender,trip_weight,origin_municipality,destination_municipality,household_id,source_person_id,stage_count,activity_destination,travel_time_min,fare_amount
0,m00000,u0002,t00000,0,-70.590951,-33.478171,-70.584048,-33.467465,2026-03-06 14:48:00+00:00,2026-03-06 16:29:00+00:00,88b2c50949fffff,88b2c50b2bfffff,metro,leisure,holiday,morning,female,3.674,Quilicura,La Florida,H00011,P00019,2,estudio,101.0,760.0
1,m00001,u0019,t00001,0,-70.702725,-33.407415,-70.639562,-33.363516,2026-03-04 07:24:00+00:00,2026-03-04 09:02:00+00:00,88b2c55717fffff,88b2c55643fffff,car,leisure,weekday,evening,male,2.245,San Miguel,Vitacura,H00020,P00021,1,salud,98.0,1520.0
2,m00002,u0016,t00002,0,-70.644146,-33.420764,-70.637746,-33.456427,2026-03-01 10:05:00+00:00,2026-03-01 12:00:00+00:00,88b2c5545bfffff,88b2c554c7fffff,bus,other,holiday,midday,other,3.384,San Miguel,Providencia,H00005,P00022,3,compra,115.0,900.0
3,m00003,u0010,t00003,0,-70.597441,-33.416981,-70.602186,-33.436174,2026-03-08 03:19:00+00:00,2026-03-08 03:36:00+00:00,88b2c5568bfffff,88b2c556bbfffff,bus,transfer,holiday,midday,unknown,0.548,Providencia,Providencia,H00000,P00007,1,salud,17.0,800.0
4,m00004,u0010,t00004,0,-70.690935,-33.415534,-70.679502,-33.443332,2026-03-02 12:52:00+00:00,2026-03-02 14:38:00+00:00,88b2c55469fffff,88b2c5543dfffff,scooter,errand,weekday,midday,male,1.441,Santiago,San Miguel,H00001,P00017,2,salud,106.0,1520.0


### D4R. selected_fields=subconjunto + keep_extra_fields=True rico

In [94]:
trips_d4r, report_d4r = import_trips_from_dataframe(
    df_d_rich,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_d4r",
    options=ImportOptions(
        selected_fields=subset_selected,
        keep_extra_fields=True,
    ),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

assert report_d4r.ok is True
assert len(trips_d4r.data) == 50

final_cols_d4r = set(trips_d4r.data.columns)
assert expected_subset_schema_cols.issubset(final_cols_d4r)

dropped_optional_schema_d4r = schema_optional_present_in_source - set(subset_selected)
assert final_cols_d4r.isdisjoint(dropped_optional_schema_d4r)

assert extra_cols_present_in_source.issubset(final_cols_d4r)

fields_eff_d4r = set(trips_d4r.metadata["schema_effective"]["fields_effective"])
assert fields_eff_d4r == expected_subset_schema_cols

assert set(trips_d4r.metadata["extra_fields_kept"]) == extra_cols_present_in_source
assert set(report_d4r.metadata["metadata"]["extra_fields_kept"]) == extra_cols_present_in_source
assert set(report_d4r.metadata["metadata"]["schema_effective"]["fields_effective"]) == expected_subset_schema_cols

event_d4r = trips_d4r.metadata["events"][-1]
assert event_d4r["summary"]["input_rows"] == 50
assert event_d4r["summary"]["output_rows"] == 50
assert set(event_d4r["summary"]["columns_deleted"]) >= dropped_optional_schema_d4r

## Sección 6. Trazabilidad y contrato final

En esta sección se verifica que `import_trips_from_dataframe(...)` no solo construya
un `TripDataset` usable, sino que además deje evidencia consistente y reproducible en:

- `TripDataset.metadata`
- `TripDataset.metadata["events"]`
- `ImportReport`
- `ImportReport.summary`
- `ImportReport.metadata`
- `schema_effective`
- `domains_effective`
- `metadata["is_validated"]`

Subbloques:

- E1. Caso feliz simple
- E2. Caso con warnings
- E3. Caso con `domains_extended`
- E4. Caso Tier 2
- E5. Caso con correspondencias

### Checks comunes

In [95]:
def assert_common_contract(trips, report, expected_rows):
    assert isinstance(trips, TripDataset)
    assert isinstance(report, ImportReport)
    assert report.ok is True

    # dataset.metadata mínimo
    assert isinstance(trips.metadata, dict)
    assert "dataset_id" in trips.metadata
    assert isinstance(trips.metadata["dataset_id"], str)
    assert trips.metadata["dataset_id"] != ""

    assert "is_validated" in trips.metadata
    assert trips.metadata["is_validated"] is False

    assert "schema" in trips.metadata
    assert "schema_effective" in trips.metadata
    assert "events" in trips.metadata
    assert isinstance(trips.metadata["events"], list)
    assert len(trips.metadata["events"]) >= 1

    # evento final
    event = trips.metadata["events"][-1]
    assert event["op"] == "import_trips"
    assert isinstance(event["parameters"], dict)
    assert isinstance(event["summary"], dict)
    assert isinstance(event["issues_summary"], dict)
    assert isinstance(event["issues_summary"]["counts"], dict)
    assert isinstance(event["issues_summary"]["by_code"], dict)

    # report
    assert isinstance(report.issues, list)
    assert isinstance(report.summary, dict)
    assert isinstance(report.parameters, dict)
    assert isinstance(report.metadata, dict)

    # coherencia report <-> dataset
    assert report.metadata["dataset_id"] == trips.metadata["dataset_id"]
    assert report.metadata["summary"]["rows_out"] == expected_rows
    assert report.summary["rows_out"] == expected_rows

### E1. Caso feliz simple

In [96]:
df_e1 = make_happy_path_minimal(
    filas=6,
    seed=42,
)

trips_e1, report_e1 = import_trips_from_dataframe(
    df_e1,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_e1",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [97]:
assert_common_contract(trips_e1, report_e1, expected_rows=6)

# metadata mínima concreta
assert trips_e1.metadata["provenance"] == BASE_PROVENANCE
assert isinstance(trips_e1.metadata["schema"], dict)
assert isinstance(trips_e1.metadata["schema_effective"], dict)
assert isinstance(trips_e1.metadata["mappings"], dict)
assert isinstance(trips_e1.metadata["domains_effective"], dict)
assert isinstance(trips_e1.metadata["domains_extended"], list)
assert isinstance(trips_e1.metadata["extra_fields_kept"], list)
assert isinstance(trips_e1.metadata["temporal"], dict)

# evento import_trips
event_e1 = trips_e1.metadata["events"][-1]
assert event_e1["summary"]["input_rows"] == 6
assert event_e1["summary"]["output_rows"] == 6
assert isinstance(event_e1["summary"]["columns_deleted"], list)

assert isinstance(trips_e1.metadata["events"][-1]["issues_summary"]["counts"], dict)
assert isinstance(trips_e1.metadata["events"][-1]["issues_summary"]["by_code"], dict)

# report.summary
assert report_e1.summary["rows_in"] == 6
assert report_e1.summary["rows_out"] == 6

assert "n_fields_mapped" in report_e1.summary
assert "n_domain_mappings_applied" in report_e1.summary
assert isinstance(report_e1.summary["n_fields_mapped"], int)
assert isinstance(report_e1.summary["n_domain_mappings_applied"], int)

# report.parameters
assert report_e1.parameters["keep_extra_fields"] is True
assert report_e1.parameters["strict"] is False
assert report_e1.parameters["strict_domains"] is False
assert report_e1.parameters["single_stage"] is False

# schema_effective
schema_eff_e1 = trips_e1.metadata["schema_effective"]
assert isinstance(schema_eff_e1["fields_effective"], list)
assert isinstance(schema_eff_e1["overrides"], dict)
assert isinstance(schema_eff_e1["domains_effective"], dict)
assert isinstance(schema_eff_e1["temporal"], dict)

# domains_effective
assert isinstance(trips_e1.metadata["domains_effective"], dict)

### E2. Caso con warnings

In [98]:
df_e2 = generate_synthetic_trip_dataframe(
    filas=10,
    seed=42,
    tier_temporal="tier_1",
    tier1_datetime_format="mixed_with_invalids",
    coord_format="numeric",
    h3_mode="provided_valid",
    trip_structure="independent",
    base_fields=["mode", "purpose"],
)

trips_e2, report_e2 = import_trips_from_dataframe(
    df_e2,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_e2",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

D:\Memoria\repos\pylondrina\src\pylondrina\importing.py:1925: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  parsed = pd.to_datetime(tmp, format="mixed", errors="coerce")
D:\Memoria\repos\pylondrina\src\pylondrina\importing.py:1925: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  parsed = pd.to_datetime(tmp, format="mixed", errors="coerce")


In [99]:
assert_common_contract(trips_e2, report_e2, expected_rows=10)

codes_e2 = [iss.code for iss in report_e2.issues]
assert "IMP.TYPE.COERCE_PARTIAL" in codes_e2

# issues profundos
for iss in report_e2.issues:
    assert hasattr(iss, "level")
    assert hasattr(iss, "code")
    assert hasattr(iss, "message")
    assert hasattr(iss, "details")

# summary e issues_summary
assert trips_e2.metadata["events"][-1]["issues_summary"]["counts"]["warning"] >= 1

event_e2 = trips_e2.metadata["events"][-1]
assert event_e2["issues_summary"]["counts"]["warning"] >= 1
assert event_e2["issues_summary"]["by_code"]["IMP.TYPE.COERCE_PARTIAL"] >= 1

# metadata temporal
temporal_e2 = trips_e2.metadata["temporal"]
assert temporal_e2["tier"] == "tier_1"
assert isinstance(temporal_e2["normalization"], dict)

# coherencia report.metadata <-> trips.metadata
assert report_e2.metadata["metadata"]["temporal"]["tier"] == trips_e2.metadata["temporal"]["tier"]
assert report_e2.metadata["metadata"]["dataset_id"] == trips_e2.metadata["dataset_id"]

### E3. Caso con domains_extended

In [100]:
df_e3 = make_extended_domains(
    filas=10,
    seed=42,
    include_noise=False,
    extra_value_domains={
        "mode": ["canon", "submodes"],
        "purpose": ["canon", "finos"],
    },
)

trips_e3, report_e3 = import_trips_from_dataframe(
    df_e3,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_e3",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [101]:
assert_common_contract(trips_e3, report_e3, expected_rows=10)

codes_e3 = [iss.code for iss in report_e3.issues]
assert "DOM.EXTENSION.APPLIED" in codes_e3

# metadata domains
assert isinstance(trips_e3.metadata["domains_extended"], list)
assert len(trips_e3.metadata["domains_extended"]) >= 1

for field in trips_e3.metadata["domains_extended"]:
    dom = trips_e3.metadata["domains_effective"][field]
    assert dom["extendable"] is True
    assert dom["extended"] is True
    assert isinstance(dom["added_values"], list)
    assert dom["n_added"] >= 1

# schema_effective profundo
schema_eff_e3 = trips_e3.metadata["schema_effective"]
for field in trips_e3.metadata["domains_extended"]:
    assert field in schema_eff_e3["overrides"]
    assert "domain_extended" in schema_eff_e3["overrides"][field]["reasons"]

# report metadata
assert report_e3.metadata["metadata"]["domains_extended"] == trips_e3.metadata["domains_extended"]

issues_count = trips_e3.metadata["events"][-1]["issues_summary"]["counts"]
assert issues_count.get("info", 0) + issues_count.get("warning", 0) >= 1

# evento
event_e3 = trips_e3.metadata["events"][-1]
assert event_e3["issues_summary"]["by_code"]["DOM.EXTENSION.APPLIED"] >= 1

### E4. Caso Tier 2

In [102]:
df_e4 = make_tier2_valid(
    filas=8,
    seed=42,
)

trips_e4, report_e4 = import_trips_from_dataframe(
    df_e4,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_e4",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [103]:
assert_common_contract(trips_e4, report_e4, expected_rows=8)

# temporal
temporal_e4 = trips_e4.metadata["temporal"]
assert temporal_e4["tier"] == "tier_2"
assert isinstance(temporal_e4["fields_present"], list)
assert "origin_time_local_hhmm" in temporal_e4["fields_present"]
assert "destination_time_local_hhmm" in temporal_e4["fields_present"]

# schema_effective.temporal
schema_eff_e4 = trips_e4.metadata["schema_effective"]
assert isinstance(schema_eff_e4["temporal"], dict)

assert trips_e4.metadata["temporal"]["tier"] == "tier_2"

# report.metadata temporal
assert report_e4.metadata["metadata"]["temporal"]["tier"] == "tier_2"

# summary
assert report_e4.summary["rows_in"] == 8
assert report_e4.summary["rows_out"] == 8

# evento
event_e4 = trips_e4.metadata["events"][-1]
assert event_e4["summary"]["input_rows"] == 8
assert event_e4["summary"]["output_rows"] == 8

### E5. Caso con correspondencias

In [104]:
field_corr_e5 = {
    "movement_id": "id_mov_fuente",
    "user_id": "id_persona_fuente",
    "origin_longitude": "lon_o_fuente",
    "origin_latitude": "lat_o_fuente",
    "destination_longitude": "lon_d_fuente",
    "destination_latitude": "lat_d_fuente",
    "origin_h3_index": "h3_o_fuente",
    "destination_h3_index": "h3_d_fuente",
    "origin_time_utc": "t_origen_fuente",
    "destination_time_utc": "t_destino_fuente",
    "trip_id": "id_viaje_fuente",
    "movement_seq": "seq_fuente",
    "mode": "modo_fuente",
    "purpose": "proposito_fuente",
}

df_e5 = make_happy_path_minimal(
    filas=8,
    seed=42,
    base_fields=["mode", "purpose"],
    field_correspondence=field_corr_e5,
)

df_e5["modo_fuente"] = ["A PIE", "BUS", "AUTO", "METRO", "A PIE", "BUS", "AUTO", "METRO"]
df_e5["proposito_fuente"] = ["TRABAJO", "ESTUDIO", "HOGAR", "COMPRAS", "TRABAJO", "ESTUDIO", "HOGAR", "COMPRAS"]

value_corr_e5 = {
    "mode": {
        "A PIE": "walk",
        "BUS": "bus",
        "AUTO": "car",
        "METRO": "metro",
    },
    "purpose": {
        "TRABAJO": "work",
        "ESTUDIO": "education",
        "HOGAR": "home",
        "COMPRAS": "shopping",
    },
}

trips_e5, report_e5 = import_trips_from_dataframe(
    df_e5,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_e5",
    options=ImportOptions(),
    field_correspondence=field_corr_e5,
    value_correspondence=value_corr_e5,
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

In [105]:
assert_common_contract(trips_e5, report_e5, expected_rows=8)

# report
assert report_e5.field_correspondence == field_corr_e5
assert report_e5.value_correspondence == value_corr_e5

# metadata.mappings
assert trips_e5.metadata["mappings"]["field_correspondence"] == field_corr_e5
assert trips_e5.metadata["mappings"]["value_correspondence"] == value_corr_e5

# objeto TripDataset
assert trips_e5.field_correspondence == field_corr_e5
assert trips_e5.value_correspondence == value_corr_e5

# evento
event_e5 = trips_e5.metadata["events"][-1]
assert event_e5["op"] == "import_trips"
assert isinstance(event_e5["parameters"], dict)
assert isinstance(event_e5["summary"], dict)
assert isinstance(event_e5["issues_summary"], dict)

assert event_e5["summary"]["input_rows"] == 8
assert event_e5["summary"]["output_rows"] == 8

# summary y metadata
assert report_e5.summary["rows_in"] == 8
assert report_e5.summary["rows_out"] == 8
assert report_e5.metadata["metadata"]["mappings"]["field_correspondence"] == field_corr_e5
assert report_e5.metadata["metadata"]["mappings"]["value_correspondence"] == value_corr_e5

### E2R. Caso con warnings "rico"

In [108]:
df_e2r = generate_synthetic_trip_dataframe(
    filas=50,
    seed=42,
    tier_temporal="tier_1",
    tier1_datetime_format="mixed_with_invalids",
    coord_format="numeric",
    h3_mode="provided_valid",
    trip_structure="independent",
    base_fields=["mode", "purpose", "day_type", "time_period", "user_gender", "trip_weight"],
    extra_columns=["household_id", "source_person_id", "stage_count", "activity_destination", "travel_time_min", "fare_amount"],
)

trips_e2r, report_e2r = import_trips_from_dataframe(
    df_e2r,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_e2r",
    options=ImportOptions(),
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

assert_common_contract(trips_e2r, report_e2r, expected_rows=50)

codes_e2r = [iss.code for iss in report_e2r.issues]
assert "IMP.TYPE.COERCE_PARTIAL" in codes_e2r

assert report_e2r.summary["rows_in"] == 50
assert report_e2r.summary["rows_out"] == 50

issues_count = trips_e2r.metadata["events"][-1]["issues_summary"]["counts"]
assert issues_count["warning"] >= 1

event_e2r = trips_e2r.metadata["events"][-1]
assert event_e2r["summary"]["input_rows"] == 50
assert event_e2r["summary"]["output_rows"] == 50
assert event_e2r["issues_summary"]["counts"]["warning"] >= 1

assert report_e2r.metadata["metadata"]["dataset_id"] == trips_e2r.metadata["dataset_id"]
assert isinstance(report_e2r.metadata["metadata"]["temporal"], dict)

D:\Memoria\repos\pylondrina\src\pylondrina\importing.py:1925: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  parsed = pd.to_datetime(tmp, format="mixed", errors="coerce")
D:\Memoria\repos\pylondrina\src\pylondrina\importing.py:1925: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  parsed = pd.to_datetime(tmp, format="mixed", errors="coerce")


### E5R. Caso con correspondencias rico

In [110]:
df_e5r = make_happy_path_rich(
    filas=50,
    seed=42,
    field_correspondence=field_corr_e5,
    base_fields=["mode", "purpose", "day_type", "time_period", "user_gender", "trip_weight"],
    extra_columns=["household_id", "source_person_id", "stage_count", "activity_destination", "travel_time_min", "fare_amount"],
)

pattern_mode = ["A PIE", "BUS", "AUTO", "METRO"] * 13
pattern_purpose = ["TRABAJO", "ESTUDIO", "HOGAR", "COMPRAS"] * 13

df_e5r["modo_fuente"] = pattern_mode[:50]
df_e5r["proposito_fuente"] = pattern_purpose[:50]

trips_e5r, report_e5r = import_trips_from_dataframe(
    df_e5r,
    schema=BASE_TRIP_SCHEMA,
    source_name="synthetic_e5r",
    options=ImportOptions(keep_extra_fields=True),
    field_correspondence=field_corr_e5,
    value_correspondence=value_corr_e5,
    provenance=BASE_PROVENANCE,
    h3_resolution=8,
)

assert_common_contract(trips_e5r, report_e5r, expected_rows=50)

assert report_e5r.field_correspondence == field_corr_e5
assert report_e5r.value_correspondence == value_corr_e5

assert trips_e5r.metadata["mappings"]["field_correspondence"] == field_corr_e5
assert trips_e5r.metadata["mappings"]["value_correspondence"] == value_corr_e5

assert report_e5r.summary["rows_in"] == 50
assert report_e5r.summary["rows_out"] == 50
assert report_e5r.metadata["metadata"]["dataset_id"] == trips_e5r.metadata["dataset_id"]

event_e5r = trips_e5r.metadata["events"][-1]
assert event_e5r["summary"]["input_rows"] == 50
assert event_e5r["summary"]["output_rows"] == 50

## Sección 7. Resumen de cobertura

### 7.1 Checklist de escenarios cubiertos
### 7.2 Bugs detectados
### 7.3 Decisiones o ajustes pendientes
### 7.4 Cobertura alcanzada
### 7.5 Ideas de migración futura a `pytest`